# COMP9444 FOMC：LSTM、BiLSTM 与 LSTM+TCN 完整实验记录

本 Notebook 是循环模型实验的完整执行和证据记录。它会展示固定 baseline、所有保留的调参尝试、没有成功的候选、额外的 LSTM+TCN 混合模型、三 seed 测试指标、混淆矩阵、学习曲线和错误案例。

Notebook 首先读取已经生成的 canonical 结果。如果结果目录缺失，初始化单元会自动使用当前 Python kernel 调用对应脚本并继续执行。因此从头到尾执行全部单元即可完成展示；如果需要重新训练，CPU 上可能耗时较长。

## 任务协议和论文对齐

任务是句子级 FOMC 政策立场三分类：0 Dovish（鸽派）、1 Hawkish（鹰派）、2 Neutral（中性）。统一协议使用固定的分层 validation split，用 validation weighted F1 选择配置，最后在 5768、78516、944601 三个 held-out seed 上测试。

Shah et al. (2023) 第 4.2 节描述的是单层 LSTM/BiLSTM baseline，包括 word tokenisation、padding masking、dropout 和 cross-entropy 训练。下面的 LSTM+TCN 明确作为额外的混合结构消融实验展示，不替换必须保留的 baseline，也不把它称为 pretrained model fine-tuning。

## 自包含实现代码

下面的代码单元内嵌了数据读取、文本预处理、Vocabulary、padding、长度 masking、LSTM/BiLSTM、LSTM+TCN、训练、评估、调参和结果生成逻辑。Notebook 不再从 src/ 或 scripts/ 导入代码；它仍需要 PyTorch 等 Python 包和原始 FOMC Excel 数据。

In [ ]:
# add
# Self-contained implementation: no imports from src/ or scripts/ are needed.

"""Train and evaluate the COMP9444 LSTM and BiLSTM models.

The script intentionally uses only the Python standard library for reading the
Excel files. This keeps the data protocol independent from pandas/openpyxl.

Typical usage from the repository root:

    python src/lstm_bilstm.py --prepare-only
    python src/lstm_bilstm.py --models lstm bilstm --seeds 5768 78516 944601

The repository's project plan remains the source of truth for the experiment
settings. The default configuration follows its recommended base path, while
``--tune`` runs the specified hyperparameter grid on seed 5768 only and then
freezes the best configuration for the requested final seeds.
"""


import argparse
import copy
import csv
import hashlib
import json
import math
import random
import re
import statistics
import sys
import time
from collections import Counter
from dataclasses import dataclass
from itertools import product
from pathlib import Path
from typing import Any, Iterable, Sequence
from zipfile import ZipFile
import xml.etree.ElementTree as ET

import numpy as np
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset


LABEL_NAMES = {0: "Dovish", 1: "Hawkish", 2: "Neutral"}
LABEL_ORDER = [0, 1, 2]
SPECIAL_TOKENS = ["<pad>", "<unk>"]
# The paper describes lowercasing, punctuation removal, and word tokenisation.
# This pattern keeps alphanumeric word pieces and drops punctuation entirely.
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+")
SHEET_NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"


@dataclass(frozen=True)
class Record:
    sample_id: str
    original_index: int
    sentence: str
    original_sentence: str
    year: int
    label: int
    source_file: str


def normalise_sentence(text: str) -> str:
    """Apply the shared whitespace normalisation without changing the words."""

    return re.sub(r"\s+", " ", text.strip())


def stable_sample_id(original_index: int, year: int, label: int, sentence: str) -> str:
    """Build an ID shared by all three seed files without trusting duplicate indices."""

    key = f"{original_index}|{year}|{label}|{sentence}".encode("utf-8")
    return f"fomc_{hashlib.sha1(key).hexdigest()}"


def column_index(cell_ref: str) -> int:
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    index = 0
    for character in letters:
        index = index * 26 + ord(character.upper()) - ord("A") + 1
    return index - 1


def cell_text(cell: ET.Element, shared_strings: list[str]) -> str:
    cell_type = cell.attrib.get("t")
    if cell_type == "inlineStr":
        return "".join(t.text or "" for t in cell.iter(SHEET_NS + "t"))

    value = cell.find(SHEET_NS + "v")
    if value is None:
        return ""
    if cell_type == "s":
        return shared_strings[int(value.text)]
    return value.text or ""


def read_xlsx(path: Path) -> list[Record]:
    """Read one checkpoint workbook using the XLSX ZIP/XML representation."""

    with ZipFile(path) as workbook:
        shared_strings: list[str] = []
        if "xl/sharedStrings.xml" in workbook.namelist():
            root = ET.fromstring(workbook.read("xl/sharedStrings.xml"))
            for item in root:
                shared_strings.append(
                    "".join(t.text or "" for t in item.iter(SHEET_NS + "t"))
                )

        sheet = ET.fromstring(workbook.read("xl/worksheets/sheet1.xml"))
        rows: list[list[str]] = []
        for row in sheet.find(SHEET_NS + "sheetData"):
            values: list[str] = []
            for cell in row:
                index = column_index(cell.attrib["r"])
                while len(values) <= index:
                    values.append("")
                values[index] = cell_text(cell, shared_strings)
            rows.append(values)

    if not rows:
        raise ValueError(f"Workbook is empty: {path}")

    headers = rows[0]
    expected = {"index", "sentence", "year", "label"}
    if not expected.issubset(headers):
        raise ValueError(f"Unexpected columns in {path.name}: {headers}")

    records: list[Record] = []
    for values in rows[1:]:
        if not any(value.strip() for value in values):
            continue
        row = {headers[i]: values[i] if i < len(values) else "" for i in range(len(headers))}
        original = row["sentence"]
        sentence = normalise_sentence(original)
        records.append(
            Record(
                sample_id=stable_sample_id(
                    int(row["index"]),
                    int(row["year"]),
                    int(row["label"]),
                    sentence,
                ),
                original_index=int(row["index"]),
                sentence=sentence,
                original_sentence=original,
                year=int(row["year"]),
                label=int(row["label"]),
                source_file=path.name,
            )
        )
    return records


def resolve_data_dir(data_dir: Path) -> Path:
    data_dir = data_dir.expanduser().resolve()
    if (data_dir / "FOMC_dataset_checkpoint").is_dir():
        data_dir = data_dir / "FOMC_dataset_checkpoint"
    required = data_dir / "lab-manual-combine-train-5768.xlsx"
    if not required.exists():
        raise FileNotFoundError(
            "Could not find the FOMC Excel files in "
            f"{data_dir}. Pass --data-dir pointing to FOMC_dataset_checkpoint."
        )
    return data_dir


def load_seed(data_dir: Path, seed: int) -> tuple[list[Record], list[Record]]:
    train_path = data_dir / f"lab-manual-combine-train-{seed}.xlsx"
    test_path = data_dir / f"lab-manual-combine-test-{seed}.xlsx"
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(f"Missing train/test workbook for seed {seed}")
    return read_xlsx(train_path), read_xlsx(test_path)


def record_to_row(record: Record, seed: int, split: str) -> dict[str, Any]:
    return {
        "sample_id": record.sample_id,
        "original_index": record.original_index,
        "sentence": record.sentence,
        "original_sentence": record.original_sentence,
        "year": record.year,
        "label": record.label,
        "source_file": record.source_file,
        "seed": seed,
        "split": split,
    }


def write_csv(path: Path, rows: Iterable[dict[str, Any]], fieldnames: Sequence[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def create_or_load_validation_split(
    train_records: list[Record],
    seed: int,
    split_dir: Path,
) -> tuple[list[Record], list[Record]]:
    """Create or validate the shared, stratified validation split for one seed."""

    train_path = split_dir / f"{seed}_train.csv"
    val_path = split_dir / f"{seed}_val.csv"
    by_id = {record.sample_id: record for record in train_records}
    expected_ids = set(by_id)

    if train_path.exists() and val_path.exists():
        def read_ids(path: Path) -> set[str]:
            with path.open(newline="", encoding="utf-8") as handle:
                return {row["sample_id"] for row in csv.DictReader(handle)}

        train_ids = read_ids(train_path)
        val_ids = read_ids(val_path)
        if train_ids & val_ids or train_ids | val_ids != expected_ids:
            raise ValueError(
                f"Existing split files for seed {seed} do not match the source training data"
            )
    else:
        indices = np.arange(len(train_records))
        labels = np.array([record.label for record in train_records])
        train_indices, val_indices = train_test_split(
            indices,
            test_size=0.20,
            random_state=seed,
            stratify=labels,
        )
        train_ids = {train_records[index].sample_id for index in train_indices}
        val_ids = {train_records[index].sample_id for index in val_indices}
        fields = [
            "sample_id",
            "original_index",
            "sentence",
            "original_sentence",
            "year",
            "label",
            "source_file",
            "seed",
            "split",
        ]
        write_csv(
            train_path,
            (record_to_row(by_id[index], seed, "train") for index in sorted(train_ids)),
            fields,
        )
        write_csv(
            val_path,
            (record_to_row(by_id[index], seed, "val") for index in sorted(val_ids)),
            fields,
        )

    return (
        [by_id[index] for index in sorted(train_ids)],
        [by_id[index] for index in sorted(val_ids)],
    )


def write_overlap_audit(
    train_records: list[Record],
    test_records: list[Record],
    seed: int,
    audit_dir: Path,
) -> dict[str, Any]:
    """Record the exact and normalised train/test overlap audit."""

    exact_train = {record.original_sentence for record in train_records}
    exact_test = {record.original_sentence for record in test_records}
    normalised_train = {record.sentence for record in train_records}
    normalised_test = {record.sentence for record in test_records}
    train_ids = {record.sample_id for record in train_records}
    test_ids = {record.sample_id for record in test_records}
    audit = {
        "seed": seed,
        "train_rows": len(train_records),
        "test_rows": len(test_records),
        "train_unique_indices": len(train_ids),
        "test_unique_indices": len(test_ids),
        "index_overlap_count": len(train_ids & test_ids),
        "exact_sentence_overlap_count": len(exact_train & exact_test),
        "normalised_sentence_overlap_count": len(normalised_train & normalised_test),
        "duplicate_sentences_within_train": len(train_records) - len(exact_train),
        "duplicate_sentences_within_test": len(test_records) - len(exact_test),
    }
    audit_dir.mkdir(parents=True, exist_ok=True)
    with (audit_dir / f"{seed}_overlap_audit.json").open("w", encoding="utf-8") as handle:
        json.dump(audit, handle, indent=2)
    return audit


def tokenize(sentence: str) -> list[str]:
    return TOKEN_PATTERN.findall(sentence.lower())


class Vocabulary:
    def __init__(self, tokens: list[str]):
        self.itos = SPECIAL_TOKENS + tokens
        self.stoi = {token: index for index, token in enumerate(self.itos)}

    @property
    def pad_index(self) -> int:
        return 0

    @property
    def unk_index(self) -> int:
        return 1

    def __len__(self) -> int:
        return len(self.itos)

    def encode(self, sentence: str, max_length: int) -> tuple[list[int], int]:
        tokens = tokenize(sentence)[:max_length]
        ids = [self.stoi.get(token, self.unk_index) for token in tokens]
        length = max(1, len(ids))
        if not ids:
            ids = [self.unk_index]
        ids.extend([self.pad_index] * (max_length - len(ids)))
        return ids, length

    def to_json(self) -> dict[str, Any]:
        return {"size": len(self), "itos": self.itos}


def build_vocabulary(records: Iterable[Record], max_vocab_size: int) -> Vocabulary:
    counts: Counter[str] = Counter()
    for record in records:
        counts.update(tokenize(record.sentence))
    available = max(0, max_vocab_size - len(SPECIAL_TOKENS))
    tokens = sorted(counts, key=lambda token: (-counts[token], token))[:available]
    return Vocabulary(tokens)


class TextDataset(Dataset[tuple[torch.Tensor, int, int, int, int]]):
    def __init__(self, records: list[Record], vocabulary: Vocabulary, max_length: int):
        self.records = records
        self.examples = []
        for position, record in enumerate(records):
            ids, length = vocabulary.encode(record.sentence, max_length)
            self.examples.append(
                (torch.tensor(ids, dtype=torch.long), length, record.label, position, record.year)
            )

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, int, int, int]:
        return self.examples[index]


def collate_batch(batch: list[tuple[torch.Tensor, int, int, int, int]]) -> tuple[torch.Tensor, ...]:
    ids, lengths, labels, sample_ids, years = zip(*batch)
    return (
        torch.stack(ids),
        torch.tensor(lengths, dtype=torch.long),
        torch.tensor(labels, dtype=torch.long),
        torch.tensor(sample_ids, dtype=torch.long),
        torch.tensor(years, dtype=torch.long),
    )


class RNNClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        dropout: float,
        bidirectional: bool,
        pooling: str = "final_hidden",
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=bidirectional,
        )
        output_size = hidden_size * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(dropout)
        self.pooling = pooling
        self.attention = nn.Linear(output_size, 1) if pooling == "attention" else None
        self.classifier = nn.Linear(output_size, len(LABEL_ORDER))
        self.bidirectional = bidirectional

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu().clamp(min=1),
            batch_first=True,
            enforce_sorted=False,
        )
        output, (hidden, _) = self.rnn(packed)
        # add
        if self.pooling in {"attention", "mean", "max"}:
            output, _ = pad_packed_sequence(
                output,
                batch_first=True,
                total_length=input_ids.size(1),
            )
            positions = torch.arange(input_ids.size(1), device=lengths.device).unsqueeze(0)
            valid_tokens = positions < lengths.unsqueeze(1)
            if self.pooling == "attention":
                scores = self.attention(output).squeeze(-1)
                scores = scores.masked_fill(~valid_tokens, torch.finfo(scores.dtype).min)
                weights = torch.softmax(scores, dim=1).unsqueeze(-1)
                representation = torch.sum(output * weights, dim=1)
            elif self.pooling == "mean":
                representation = (output * valid_tokens.unsqueeze(-1)).sum(dim=1)
                representation = representation / lengths.clamp(min=1).unsqueeze(1)
            else:
                masked_output = output.masked_fill(
                    ~valid_tokens.unsqueeze(-1), -1e4
                )
                representation = masked_output.max(dim=1).values
        elif self.bidirectional:
            representation = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            representation = hidden[-1]
        #done
        return self.classifier(self.dropout(representation))


def load_glove_embeddings(
    vocabulary: Vocabulary,
    glove_path: Path,
    embedding_dim: int,
) -> tuple[torch.Tensor, int]:
    """Load matching GloVe rows and leave unmatched tokens randomly initialised."""

    rng = np.random.default_rng(9444)
    matrix = rng.normal(0.0, 0.05, size=(len(vocabulary), embedding_dim)).astype(np.float32)
    matrix[vocabulary.pad_index] = 0.0
    matched = 0
    with glove_path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            fields = line.rstrip().split()
            if len(fields) != embedding_dim + 1:
                continue
            token = fields[0]
            index = vocabulary.stoi.get(token)
            if index is None:
                continue
            matrix[index] = np.asarray(fields[1:], dtype=np.float32)
            matched += 1
    return torch.from_numpy(matrix), matched


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def choose_device(requested: str) -> torch.device:
    if requested == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device(requested)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is not available")
    return device


def make_loader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        collate_fn=collate_batch,
        generator=generator,
    )


def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    criterion: nn.Module,
) -> dict[str, Any]:
    model.eval()
    total_loss = 0.0
    total_items = 0
    labels: list[int] = []
    predictions: list[int] = []
    probabilities: list[list[float]] = []
    sample_positions: list[int] = []
    years: list[int] = []
    with torch.no_grad():
        for input_ids, lengths, batch_labels, batch_ids, batch_years in loader:
            input_ids = input_ids.to(device)
            lengths = lengths.to(device)
            batch_labels = batch_labels.to(device)
            logits = model(input_ids, lengths)
            loss = criterion(logits, batch_labels)
            batch_size = batch_labels.size(0)
            total_loss += float(loss.item()) * batch_size
            total_items += batch_size
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probabilities.extend(probs.tolist())
            predictions.extend(np.argmax(probs, axis=1).tolist())
            labels.extend(batch_labels.cpu().tolist())
            sample_positions.extend(batch_ids.tolist())
            years.extend(batch_years.tolist())

    return {
        "loss": total_loss / max(1, total_items),
        "labels": labels,
        "predictions": predictions,
        "probabilities": probabilities,
        "sample_positions": sample_positions,
        "years": years,
        "accuracy": accuracy_score(labels, predictions),
        "weighted_f1": f1_score(labels, predictions, average="weighted", zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "per_class_f1": f1_score(
            labels,
            predictions,
            labels=LABEL_ORDER,
            average=None,
            zero_division=0,
        ).tolist(),
    }


def train_model(
    model: RNNClassifier,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    learning_rate: float,
    weight_decay: float,
    scheduler_patience: int,
    max_epochs: int,
    patience: int | None,
    optimizer_name: str = "adamw",
    use_scheduler: bool = True,
    class_weights: Sequence[float] | None = None,
    gradient_clip_norm: float | None = 5.0,
) -> tuple[RNNClassifier, list[dict[str, Any]], dict[str, Any]]:
    criterion = nn.CrossEntropyLoss(
        weight=None
        if class_weights is None
        else torch.tensor(class_weights, dtype=torch.float32, device=device),
    )
    optimizer_class = (
        torch.optim.Adam
        if optimizer_name.lower() == "adam"
        else torch.optim.AdamW
    )
    optimizer = optimizer_class(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=scheduler_patience,
            min_lr=1e-6,
        )
        if use_scheduler
        else None
    )
    history: list[dict[str, Any]] = []
    best_score = -math.inf
    best_epoch = 0
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0

    model.to(device)
    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        total_items = 0
        for input_ids, lengths, labels, _, _ in train_loader:
            input_ids = input_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            if gradient_clip_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_norm)
            optimizer.step()
            batch_size = labels.size(0)
            total_loss += float(loss.item()) * batch_size
            total_items += batch_size

        validation = evaluate_model(model, val_loader, device, criterion)
        row = {
            "epoch": epoch,
            "train_loss": total_loss / max(1, total_items),
            "val_loss": validation["loss"],
            "val_accuracy": validation["accuracy"],
            "val_weighted_f1": validation["weighted_f1"],
            "val_macro_f1": validation["macro_f1"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        if scheduler is not None:
            scheduler.step(row["val_weighted_f1"])
        if row["val_weighted_f1"] > best_score + 1e-9:
            best_score = row["val_weighted_f1"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if patience is not None and epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, history, {"best_epoch": best_epoch, "best_val_weighted_f1": best_score}


def json_default(value: Any) -> Any:
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Not JSON serialisable: {type(value).__name__}")


def save_classification_report(path: Path, labels: list[int], predictions: list[int]) -> None:
    report = classification_report(
        labels,
        predictions,
        labels=LABEL_ORDER,
        target_names=[LABEL_NAMES[i] for i in LABEL_ORDER],
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for name, values in report.items():
        if isinstance(values, dict):
            rows.append({
                "class": name,
                "precision": values.get("precision", ""),
                "recall": values.get("recall", ""),
                "f1_score": values.get("f1-score", ""),
                "support": values.get("support", ""),
            })
        else:
            rows.append({
                "class": name,
                "precision": "",
                "recall": "",
                "f1_score": values,
                "support": len(labels),
            })
    write_csv(path, rows, ["class", "precision", "recall", "f1_score", "support"])


def save_confusion_matrices(
    artifact_dir: Path,
    labels: list[int],
    predictions: list[int],
) -> None:
    raw = confusion_matrix(labels, predictions, labels=LABEL_ORDER)
    denominators = raw.sum(axis=1, keepdims=True)
    normalised = np.divide(
        raw,
        denominators,
        out=np.zeros_like(raw, dtype=float),
        where=denominators != 0,
    )
    fields = ["true_label"] + [LABEL_NAMES[i] for i in LABEL_ORDER]
    write_csv(
        artifact_dir / "confusion_matrix_raw.csv",
        (
            {"true_label": LABEL_NAMES[label], **{LABEL_NAMES[col]: int(raw[row, col]) for col in LABEL_ORDER}}
            for row, label in enumerate(LABEL_ORDER)
        ),
        fields,
    )
    write_csv(
        artifact_dir / "confusion_matrix_normalized.csv",
        (
            {"true_label": LABEL_NAMES[label], **{LABEL_NAMES[col]: float(normalised[row, col]) for col in LABEL_ORDER}}
            for row, label in enumerate(LABEL_ORDER)
        ),
        fields,
    )
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        figure, axis = plt.subplots(figsize=(5.5, 4.5))
        image = axis.imshow(normalised, cmap="Blues", vmin=0.0, vmax=1.0)
        axis.set_xticks(LABEL_ORDER, [LABEL_NAMES[i] for i in LABEL_ORDER])
        axis.set_yticks(LABEL_ORDER, [LABEL_NAMES[i] for i in LABEL_ORDER])
        axis.set_xlabel("Predicted label")
        axis.set_ylabel("True label")
        axis.set_title("Normalised confusion matrix")
        for row in LABEL_ORDER:
            for col in LABEL_ORDER:
                axis.text(col, row, f"{normalised[row, col]:.2f}", ha="center", va="center")
        figure.colorbar(image, ax=axis)
        figure.tight_layout()
        figure.savefig(artifact_dir / "confusion_matrix_normalized.png", dpi=180)
        plt.close(figure)
    except ImportError:
        pass


def save_learning_curve(
    path: Path,
    history: list[dict[str, Any]],
    title: str,
    best_epoch: int,
) -> None:
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        epochs = [row["epoch"] for row in history]
        figure, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].plot(epochs, [row["train_loss"] for row in history], label="train loss")
        axes[0].plot(epochs, [row["val_loss"] for row in history], label="validation loss")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Cross-entropy loss")
        axes[0].legend()
        axes[1].plot(epochs, [row["val_weighted_f1"] for row in history], label="validation weighted F1")
        axes[1].plot(epochs, [row["val_macro_f1"] for row in history], label="validation macro F1")
        axes[1].axvline(best_epoch, color="black", linestyle="--", label=f"best epoch {best_epoch}")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("F1")
        axes[1].legend()
        figure.suptitle(title)
        figure.tight_layout()
        figure.savefig(path, dpi=180)
        plt.close(figure)
    except ImportError:
        pass


def error_categories(sentence: str, true_label: int, predicted_label: int) -> str:
    text = sentence.lower()
    categories: list[str] = []
    if re.search(r"\b(no|not|never|without|unlikely|less|lower|decline|declined)\b", text):
        categories.append("negation_or_downward_term")
    if re.search(r"\b(but|however|although|while|despite|yet)\b", text):
        categories.append("mixed_tone_or_contrast")
    if len(tokenize(sentence)) >= 50:
        categories.append("long_sentence")
    if re.search(r"\b(inflation|prices?|price pressure|employment|unemployment|wages?|labor|labour|jobs?)\b", text):
        categories.append("inflation_or_employment_direction")
    if true_label == 2 or predicted_label == 2:
        categories.append("neutral_class_error")
    return ";".join(categories) or "other_error"


def save_predictions_and_errors(
    artifact_dir: Path,
    records: list[Record],
    evaluation: dict[str, Any],
    model_name: str,
    seed: int,
) -> None:
    rows = []
    error_rows = []
    for sample_id, year, label, prediction, probabilities in (
        zip(
            evaluation["sample_positions"],
            evaluation["years"],
            evaluation["labels"],
            evaluation["predictions"],
            evaluation["probabilities"],
        )
    ):
        record = records[sample_id]
        row = {
            "sample_id": record.sample_id,
            "original_index": record.original_index,
            "sentence": record.sentence,
            "source_file": record.source_file,
            "year": year,
            "true_label": label,
            "true_label_name": LABEL_NAMES[label],
            "predicted_label": prediction,
            "predicted_label_name": LABEL_NAMES[prediction],
            "prob_dovish": probabilities[0],
            "prob_hawkish": probabilities[1],
            "prob_neutral": probabilities[2],
            "correct": int(label == prediction),
            "seed": seed,
            "model_name": model_name,
        }
        rows.append(row)
        if label != prediction:
            error_rows.append({
                **row,
                "category": error_categories(record.sentence, label, prediction),
                "word_count": len(tokenize(record.sentence)),
            })
    fields = list(rows[0]) if rows else ["sample_id", "sentence"]
    write_csv(artifact_dir / "predictions.csv", rows, fields)
    error_fields = list(error_rows[0]) if error_rows else ["sample_id", "sentence", "category"]
    write_csv(artifact_dir / "error_analysis.csv", error_rows, error_fields)
    write_csv(artifact_dir / "selected_error_cases.csv", error_rows[:10], error_fields)


def save_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, default=json_default)


def make_model(model_name: str, vocabulary: Vocabulary, config: dict[str, Any]) -> RNNClassifier:
    bidirectional = model_name == "bilstm"
    model = RNNClassifier(
        vocab_size=len(vocabulary),
        embedding_dim=config["embedding_dim"],
        hidden_size=config["hidden_size"],
        dropout=config["dropout"],
        bidirectional=bidirectional,
        pooling=config.get("pooling", "final_hidden"),
    )
    glove_path = config.get("glove_path")
    if glove_path:
        if model_name != "bilstm":
            raise ValueError("GloVe comparison is only enabled for BiLSTM")
        embeddings, matched = load_glove_embeddings(
            vocabulary,
            Path(glove_path),
            config["embedding_dim"],
        )
        model.embedding.weight.data.copy_(embeddings)
        # add
        freeze_embeddings = config.get("freeze_embeddings", True)
        model.embedding.weight.requires_grad = not freeze_embeddings
        config["glove_matched_tokens"] = matched
        config["freeze_embeddings"] = freeze_embeddings
        #done
    return model


def run_one(
    model_name: str,
    seed: int,
    data_dir: Path,
    split_dir: Path,
    output_dir: Path,
    config: dict[str, Any],
    device: torch.device,
    save_artifacts: bool = True,
    evaluate_test: bool = True,
) -> dict[str, Any]:
    set_seed(seed)
    train_records, test_records = load_seed(data_dir, seed)
    split_train, val_records = create_or_load_validation_split(train_records, seed, split_dir)
    audit = write_overlap_audit(train_records, test_records, seed, split_dir.parent / "audits")
    vocabulary = build_vocabulary(split_train, config["max_vocab_size"])
    configured_max_length = config.get("max_length")
    if configured_max_length in (None, "train_max", "longest_train"):
        max_length = max(len(tokenize(record.sentence)) for record in split_train)
    else:
        max_length = int(configured_max_length)
    train_dataset = TextDataset(split_train, vocabulary, max_length)
    val_dataset = TextDataset(val_records, vocabulary, max_length)
    test_dataset = TextDataset(test_records, vocabulary, max_length)
    train_loader = make_loader(train_dataset, config["batch_size"], True, seed)
    val_loader = make_loader(val_dataset, config["batch_size"], False, seed)
    test_loader = make_loader(test_dataset, config["batch_size"], False, seed)
    model = make_model(model_name, vocabulary, config)
    training_started = time.perf_counter()
    model, history, best = train_model(
        model,
        train_loader,
        val_loader,
        device,
        config["learning_rate"],
        config["weight_decay"],
        config["scheduler_patience"],
        config["max_epochs"],
        config["early_stopping_patience"],
        optimizer_name=config.get("optimizer", "adamw"),
        use_scheduler=config.get("use_scheduler", True),
        class_weights=config.get("class_weights"),
        gradient_clip_norm=config.get("gradient_clip_norm", 5.0),
    )
    training_seconds = time.perf_counter() - training_started
    criterion = nn.CrossEntropyLoss()
    inference_started = time.perf_counter()
    validation = evaluate_model(model, val_loader, device, criterion)
    test = evaluate_model(model, test_loader, device, criterion) if evaluate_test else None
    inference_seconds = time.perf_counter() - inference_started
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    result = {
        "model": model_name,
        "seed": seed,
        "validation": {
            "loss": validation["loss"],
            "accuracy": validation["accuracy"],
            "weighted_f1": validation["weighted_f1"],
            "macro_f1": validation["macro_f1"],
            "per_class_f1": validation["per_class_f1"],
        },
        "test": None if test is None else {
            "loss": test["loss"],
            "accuracy": test["accuracy"],
            "weighted_f1": test["weighted_f1"],
            "macro_f1": test["macro_f1"],
            "per_class_f1": test["per_class_f1"],
        },
        "best_epoch": best["best_epoch"],
        "best_val_weighted_f1": best["best_val_weighted_f1"],
        "train_rows": len(split_train),
        "validation_rows": len(val_records),
        "test_rows": len(test_records),
        "vocabulary_size": len(vocabulary),
        "configured_max_length": configured_max_length,
        "resolved_max_length": max_length,
        "train_max_tokens": max(len(tokenize(record.sentence)) for record in split_train),
        "test_truncated_rows": sum(
            len(tokenize(record.sentence)) > max_length for record in test_records
        ),
        "parameter_count": parameter_count,
        "training_seconds": training_seconds,
        "inference_seconds": inference_seconds,
        "overlap_audit": audit,
    }
    if not save_artifacts:
        return result

    artifact_dir = output_dir / model_name / f"seed_{seed}"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    run_config = {
        **config,
        "model": model_name,
        "model_name": model_name,
        "seed": seed,
        "random_seed": seed,
        "epochs": config["max_epochs"],
        "configured_max_length": configured_max_length,
        "resolved_max_length": max_length,
        "best_epoch": best["best_epoch"],
        "best_val_weighted_f1": best["best_val_weighted_f1"],
        "data_dir": str(data_dir),
        "split_dir": str(split_dir),
        "output_dir": str(artifact_dir),
        "device": str(device),
        "vocabulary": vocabulary.to_json(),
        "label_names": LABEL_NAMES,
        "pretrained_checkpoint": config.get("glove_path"),
        "checkpoint_revision": None,
        "training_type": "static_pretrained_embedding" if config.get("glove_path") else "random_embedding",
        "tokenizer": "word-level-regex",
        "train_split_file": str(split_dir / f"{seed}_train.csv"),
        "validation_split_file": str(split_dir / f"{seed}_val.csv"),
        "test_split_file": str(data_dir / f"lab-manual-combine-test-{seed}.xlsx"),
        "library_versions": {
            "python": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
            "numpy": np.__version__,
            "torch": torch.__version__,
        },
    }
    save_json(artifact_dir / "config.json", run_config)
    save_json(artifact_dir / "metrics.json", result)
    write_csv(artifact_dir / "training_history.csv", history, list(history[0]) if history else ["epoch"])
    if test is not None:
        save_classification_report(artifact_dir / "classification_report.csv", test["labels"], test["predictions"])
        save_confusion_matrices(artifact_dir, test["labels"], test["predictions"])
        save_predictions_and_errors(artifact_dir, test_records, test, model_name, seed)
    save_learning_curve(
        artifact_dir / "learning_curve.png",
        history,
        f"{model_name.upper()} seed {seed}",
        best["best_epoch"],
    )
    torch.save(model.state_dict(), artifact_dir / "best_model.pt")
    return result


def tuning_grid(args: argparse.Namespace) -> list[dict[str, Any]]:
    values = product(
        args.tuning_embedding_dims,
        args.tuning_hidden_sizes,
        args.tuning_dropouts,
        args.tuning_learning_rates,
        args.tuning_batch_sizes,
        args.tuning_optimizers,
        args.tuning_poolings,
    )
    configs = []
    for (
        embedding_dim,
        hidden_size,
        dropout,
        learning_rate,
        batch_size,
        optimizer,
        pooling,
    ) in values:
        configs.append({
            "max_vocab_size": args.max_vocab_size,
            "max_length": args.max_length,
            "embedding_dim": embedding_dim,
            "hidden_size": hidden_size,
            "dropout": dropout,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "weight_decay": args.weight_decay,
            "scheduler_patience": args.scheduler_patience,
            "max_epochs": args.tuning_epochs,
            "early_stopping_patience": args.patience,
            "optimizer": optimizer,
            "pooling": pooling,
            "use_scheduler": args.use_scheduler,
            "gradient_clip_norm": args.gradient_clip_norm,
        })
    return configs[: args.max_tuning_runs]


def tune_model(
    model_name: str,
    args: argparse.Namespace,
    data_dir: Path,
    split_dir: Path,
    device: torch.device,
) -> dict[str, Any]:
    rows = []
    configs = tuning_grid(args)
    for run_number, config in enumerate(configs, start=1):
        started = time.time()
        result = run_one(
            model_name,
            5768,
            data_dir,
            split_dir,
            args.output_dir,
            config,
            device,
            save_artifacts=False,
            evaluate_test=False,
        )
        rows.append({
            "run": run_number,
            "model": model_name,
            "validation_weighted_f1": result["validation"]["weighted_f1"],
            "validation_macro_f1": result["validation"]["macro_f1"],
            "seconds": time.time() - started,
            **config,
        })
        print(
            f"tuning {model_name} {run_number}/{len(configs)}: "
            f"val weighted F1={rows[-1]['validation_weighted_f1']:.4f}"
        )
    if not rows:
        raise ValueError("No tuning configurations were generated")
    rows.sort(key=lambda row: row["validation_weighted_f1"], reverse=True)
    save_json(args.output_dir / f"best_{model_name}_config.json", rows[0])
    write_csv(args.output_dir / f"tuning_{model_name}.csv", rows, list(rows[0]))
    return {
        key: rows[0][key]
        for key in (
            "max_vocab_size",
            "max_length",
            "embedding_dim",
            "hidden_size",
            "dropout",
            "learning_rate",
            "batch_size",
            "weight_decay",
            "scheduler_patience",
            "optimizer",
            "pooling",
            "use_scheduler",
            "gradient_clip_norm",
        )
    }


def run_validation_grid(
    model_name: str,
    configs: Sequence[dict[str, Any]],
    data_dir: Path,
    split_dir: Path,
    output_dir: Path,
    device: torch.device,
    selection_seed: int = 5768,
    name: str | None = None,
) -> list[dict[str, Any]]:
    """Evaluate configurations on one validation split without touching test data."""

    rows: list[dict[str, Any]] = []
    for run_number, config in enumerate(configs, start=1):
        started = time.perf_counter()
        result = run_one(
            model_name=model_name,
            seed=selection_seed,
            data_dir=data_dir,
            split_dir=split_dir,
            output_dir=output_dir,
            config=dict(config),
            device=device,
            save_artifacts=False,
            evaluate_test=False,
        )
        rows.append({
            "run": run_number,
            "model": model_name,
            "validation_weighted_f1": result["validation"]["weighted_f1"],
            "validation_macro_f1": result["validation"]["macro_f1"],
            "validation_accuracy": result["validation"]["accuracy"],
            "resolved_max_length": result["resolved_max_length"],
            "parameter_count": result["parameter_count"],
            "seconds": time.perf_counter() - started,
            **config,
        })
        print(
            f"grid {model_name} {run_number}/{len(configs)}: "
            f"validation weighted F1={rows[-1]['validation_weighted_f1']:.4f}"
        )
    if not rows:
        raise ValueError("No validation configurations were supplied")
    rows.sort(key=lambda row: row["validation_weighted_f1"], reverse=True)
    prefix = name or model_name
    save_json(output_dir / f"best_{prefix}_config.json", rows[0])
    write_csv(output_dir / f"validation_grid_{prefix}.csv", rows, list(rows[0]))
    return rows


def save_summary(output_dir: Path, results: list[dict[str, Any]]) -> None:
    rows = []
    for result in results:
        rows.append({
            "model": result["model"],
            "seed": result["seed"],
            "validation_weighted_f1": result["validation"]["weighted_f1"],
            "validation_macro_f1": result["validation"]["macro_f1"],
            "test_accuracy": result["test"]["accuracy"],
            "test_weighted_f1": result["test"]["weighted_f1"],
            "test_macro_f1": result["test"]["macro_f1"],
            "dovish_f1": result["test"]["per_class_f1"][0],
            "hawkish_f1": result["test"]["per_class_f1"][1],
            "neutral_f1": result["test"]["per_class_f1"][2],
            "training_seconds": result["training_seconds"],
            "inference_seconds": result["inference_seconds"],
            "parameter_count": result["parameter_count"],
            "best_epoch": result["best_epoch"],
        })
    if not rows:
        return
    write_csv(output_dir / "lstm_bilstm_summary.csv", rows, list(rows[0]))
    aggregate = {}
    for model in sorted({row["model"] for row in rows}):
        model_rows = [row for row in rows if row["model"] == model]
        aggregate[model] = {}
        for metric in (
            "test_weighted_f1",
            "test_macro_f1",
            "test_accuracy",
            "dovish_f1",
            "hawkish_f1",
            "neutral_f1",
        ):
            values = [float(row[metric]) for row in model_rows]
            aggregate[model][metric] = {
                "mean": statistics.mean(values),
                "std": statistics.stdev(values) if len(values) > 1 else 0.0,
                "values": values,
            }
        aggregate[model]["mean_training_seconds"] = statistics.mean(
            float(row["training_seconds"]) for row in model_rows
        )
        aggregate[model]["parameter_count"] = model_rows[0]["parameter_count"]
    save_json(output_dir / "aggregate_metrics.json", aggregate)
    for model, model_metrics in aggregate.items():
        save_json(output_dir / model / "aggregate_metrics.json", model_metrics)
    for metric in ("test_weighted_f1", "test_macro_f1"):
        grouped: dict[str, list[float]] = {}
        for row in rows:
            grouped.setdefault(row["model"], []).append(float(row[metric]))
        save_json(
            output_dir / f"{metric}_aggregate.json",
            {
                model: {
                    "mean": statistics.mean(values),
                    "std": statistics.stdev(values) if len(values) > 1 else 0.0,
                    "values": values,
                }
                for model, values in grouped.items()
            },
        )
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        models = sorted({row["model"] for row in rows})
        seeds = sorted({row["seed"] for row in rows})
        x = np.arange(len(seeds))
        width = 0.8 / max(1, len(models))
        for metric, filename, title in (
            ("test_weighted_f1", "lstm_bilstm_weighted_f1.png", "LSTM vs BiLSTM weighted F1"),
            ("test_macro_f1", "lstm_bilstm_macro_f1.png", "LSTM vs BiLSTM macro F1"),
        ):
            figure, axis = plt.subplots(figsize=(8, 4.5))
            for offset, model in enumerate(models):
                values = [
                    next(row[metric] for row in rows if row["model"] == model and row["seed"] == seed)
                    for seed in seeds
                ]
                axis.bar(x + (offset - (len(models) - 1) / 2) * width, values, width, label=model.upper())
            axis.set_xticks(x, [str(seed) for seed in seeds])
            axis.set_xlabel("Seed")
            axis.set_ylabel("F1")
            axis.set_ylim(0, 1)
            axis.set_title(title)
            axis.legend()
            figure.tight_layout()
            figure.savefig(output_dir / filename, dpi=180)
            plt.close(figure)
    except ImportError:
        pass


def parse_args() -> argparse.Namespace:
    repo_root = Path(__file__).resolve().parents[1]
    default_data_dir = repo_root.parent / "FOMC_dataset_checkpoint"
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-dir", type=Path, default=default_data_dir)
    parser.add_argument("--split-dir", type=Path, default=repo_root / "data" / "splits")
    parser.add_argument("--output-dir", type=Path, default=repo_root / "results" / "lstm_bilstm")
    parser.add_argument("--models", nargs="+", choices=["lstm", "bilstm"], default=["lstm", "bilstm"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[5768, 78516, 944601])
    parser.add_argument("--max-vocab-size", type=int, default=10000)
    parser.add_argument(
        "--max-length",
        default="200",
        help="Integer token limit, or 'train_max' for the paper's longest-train sentence.",
    )
    parser.add_argument("--embedding-dim", type=int, default=100)
    parser.add_argument("--hidden-size", type=int, default=128)
    parser.add_argument("--dropout", type=float, default=0.5)
    parser.add_argument("--learning-rate", type=float, default=1e-3)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--optimizer", choices=["adam", "adamw"], default="adamw")
    parser.add_argument("--pooling", choices=["final_hidden", "attention"], default="final_hidden")
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--scheduler-patience", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--patience", type=int, default=6)
    parser.add_argument("--tuning-epochs", type=int, default=20)
    parser.add_argument("--tuning-embedding-dims", nargs="+", type=int, default=[100, 200])
    parser.add_argument("--tuning-hidden-sizes", nargs="+", type=int, default=[64, 128])
    parser.add_argument("--tuning-dropouts", nargs="+", type=float, default=[0.3, 0.5])
    parser.add_argument("--tuning-learning-rates", nargs="+", type=float, default=[1e-3, 3e-4])
    parser.add_argument("--tuning-batch-sizes", nargs="+", type=int, default=[16, 32])
    parser.add_argument("--tuning-optimizers", nargs="+", choices=["adam", "adamw"], default=["adamw"])
    parser.add_argument("--tuning-poolings", nargs="+", choices=["final_hidden", "attention"], default=["final_hidden"])
    parser.add_argument("--use-scheduler", action="store_true", default=True)
    parser.add_argument("--no-scheduler", dest="use_scheduler", action="store_false")
    parser.add_argument("--gradient-clip-norm", type=float, default=5.0)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--glove-path", type=Path)
    tuning = parser.add_mutually_exclusive_group()
    tuning.add_argument("--tune", dest="tune", action="store_true")
    tuning.add_argument("--no-tune", dest="tune", action="store_false")
    parser.set_defaults(tune=True)
    parser.add_argument("--max-tuning-runs", type=int, default=32)
    parser.add_argument("--prepare-only", action="store_true")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    args.data_dir = resolve_data_dir(args.data_dir)
    args.split_dir = args.split_dir.expanduser().resolve()
    args.output_dir = args.output_dir.expanduser().resolve()
    device = choose_device(args.device)
    args.output_dir.mkdir(parents=True, exist_ok=True)

    for seed in args.seeds:
        train_records, test_records = load_seed(args.data_dir, seed)
        split_train, val_records = create_or_load_validation_split(
            train_records, seed, args.split_dir
        )
        audit = write_overlap_audit(
            train_records,
            test_records,
            seed,
            args.split_dir.parent / "audits",
        )
        print(
            f"seed {seed}: train={len(split_train)}, val={len(val_records)}, "
            f"test={len(test_records)}, index_overlap={audit['index_overlap_count']}, "
            f"normalised_text_overlap={audit['normalised_sentence_overlap_count']}"
        )

    if args.prepare_only:
        print(f"Prepared shared splits and audits in {args.split_dir.parent}")
        return

    base_config = {
        "max_vocab_size": args.max_vocab_size,
        "max_length": args.max_length,
        "embedding_dim": args.embedding_dim,
        "hidden_size": args.hidden_size,
        "dropout": args.dropout,
        "learning_rate": args.learning_rate,
        "batch_size": args.batch_size,
        "weight_decay": args.weight_decay,
        "scheduler_patience": args.scheduler_patience,
        "max_epochs": args.epochs,
        "early_stopping_patience": args.patience,
        "optimizer": args.optimizer,
        "use_scheduler": args.use_scheduler,
        "gradient_clip_norm": args.gradient_clip_norm,
        "pooling": args.pooling,
    }
    configs = {model: dict(base_config) for model in args.models}
    if args.glove_path:
        configs["bilstm"]["glove_path"] = str(args.glove_path.expanduser().resolve())
    if args.tune:
        for model in args.models:
            configs[model].update(tune_model(model, args, args.data_dir, args.split_dir, device))

    for model in args.models:
        save_json(
            args.output_dir / model / "config.json",
            {
                **configs[model],
                "model_name": model,
                "epochs": configs[model]["max_epochs"],
                "pretrained_checkpoint": configs[model].get("glove_path"),
                "checkpoint_revision": None,
                "training_type": "static_pretrained_embedding" if configs[model].get("glove_path") else "random_embedding",
                "tokenizer": "word-level-regex",
                "random_seed": args.seeds,
                "train_split_file_pattern": str(args.split_dir / "{seed}_train.csv"),
                "validation_split_file_pattern": str(args.split_dir / "{seed}_val.csv"),
                "test_split_file_pattern": str(args.data_dir / "lab-manual-combine-test-{seed}.xlsx"),
                "hardware": str(device),
                "library_versions": {
                    "numpy": np.__version__,
                    "torch": torch.__version__,
                },
            },
        )

    results = []
    for model in args.models:
        for seed in args.seeds:
            print(f"training {model} seed {seed} on {device}")
            results.append(
                run_one(
                    model,
                    seed,
                    args.data_dir,
                    args.split_dir,
                    args.output_dir,
                    configs[model],
                    device,
                )
            )
            latest = results[-1]
            print(
                f"  test accuracy={latest['test']['accuracy']:.4f}, "
                f"weighted F1={latest['test']['weighted_f1']:.4f}, "
                f"macro F1={latest['test']['macro_f1']:.4f}"
            )
    save_summary(args.output_dir, results)
    print(f"Saved LSTM/BiLSTM results in {args.output_dir}")

# add
"""LSTM plus temporal convolution classifier for the FOMC sentence task."""


import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class TCNResidualBlock(nn.Module):
    """A same-length residual temporal convolution block."""

    def __init__(
        self,
        input_channels: int,
        output_channels: int,
        kernel_size: int,
        dilation: int,
        dropout: float,
        normalization: str = "none",
    ) -> None:
        super().__init__()
        if kernel_size < 3 or kernel_size % 2 == 0:
            raise ValueError("TCN kernel_size must be an odd integer >= 3")
        if normalization not in {"none", "channel"}:
            raise ValueError(f"Unsupported TCN normalization: {normalization}")
        padding = dilation * (kernel_size - 1) // 2
        self.conv1 = nn.Conv1d(
            input_channels,
            output_channels,
            kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.conv2 = nn.Conv1d(
            output_channels,
            output_channels,
            kernel_size,
            padding=padding,
            dilation=dilation,
        )
        # add
        self.norm1 = (
            nn.Identity()
            if normalization == "none"
            else nn.GroupNorm(1, output_channels)
        )
        self.norm2 = (
            nn.Identity()
            if normalization == "none"
            else nn.GroupNorm(1, output_channels)
        )
        #done
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.residual = (
            nn.Identity()
            if input_channels == output_channels
            else nn.Conv1d(input_channels, output_channels, kernel_size=1)
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        residual = self.residual(inputs)
        # add
        outputs = self.norm1(self.conv1(inputs))
        outputs = self.dropout(self.activation(outputs))
        outputs = self.norm2(self.conv2(outputs))
        outputs = self.dropout(self.activation(outputs))
        #done
        return self.activation(outputs + residual)


class LSTMTCNClassifier(nn.Module):
    """Use LSTM for sequence context and TCN blocks for local patterns."""

    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        dropout: float,
        lstm_layers: int = 1,
        tcn_channels: int = 64,
        tcn_kernel_size: int = 3,
        tcn_layers: int = 2,
        pooling: str = "mean_max",
        normalization: str = "none",
        num_classes: int = 3,
    ) -> None:
        super().__init__()
        if lstm_layers < 1 or tcn_layers < 1:
            raise ValueError("lstm_layers and tcn_layers must be positive")
        if pooling not in {"mean", "max", "last", "mean_max"}:
            raise ValueError(f"Unsupported pooling: {pooling}")

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        blocks: list[nn.Module] = []
        input_channels = hidden_size
        for layer_index in range(tcn_layers):
            blocks.append(
                TCNResidualBlock(
                    input_channels=input_channels,
                    output_channels=tcn_channels,
                    kernel_size=tcn_kernel_size,
                    dilation=2**layer_index,
                    dropout=dropout,
                    normalization=normalization,
                )
            )
            input_channels = tcn_channels
        self.tcn = nn.ModuleList(blocks)
        self.pooling = pooling
        pooled_size = tcn_channels * (2 if pooling == "mean_max" else 1)
        self.output_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(pooled_size, num_classes)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu().clamp(min=1),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_output, _ = self.rnn(packed)
        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=input_ids.size(1),
        )

        positions = torch.arange(input_ids.size(1), device=lengths.device).unsqueeze(0)
        valid_tokens = positions < lengths.unsqueeze(1)
        mask = valid_tokens.unsqueeze(1).to(output.dtype)
        features = output.transpose(1, 2) * mask
        for block in self.tcn:
            features = block(features) * mask

        if self.pooling == "mean":
            representation = features.sum(dim=2) / lengths.clamp(min=1).unsqueeze(1)
        elif self.pooling == "max":
            masked_features = features.masked_fill(~valid_tokens.unsqueeze(1), -1e4)
            representation = masked_features.max(dim=2).values
        elif self.pooling == "last":
            last_positions = (lengths.clamp(min=1) - 1).view(-1, 1, 1)
            last_positions = last_positions.expand(-1, features.size(1), -1)
            representation = features.gather(2, last_positions).squeeze(2)
        else:
            mean_features = features.sum(dim=2) / lengths.clamp(min=1).unsqueeze(1)
            masked_features = features.masked_fill(~valid_tokens.unsqueeze(1), -1e4)
            max_features = masked_features.max(dim=2).values
            representation = torch.cat((mean_features, max_features), dim=1)

        return self.classifier(self.output_dropout(representation))


#done

# add
"""Tune and evaluate an additional LSTM+TCN FOMC classifier."""


import argparse
import csv
import json
import os
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch import nn


OFFICIAL_SEEDS = (5768, 78516, 944601)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--repo-root",
        type=Path,
        default=Path(__file__).resolve().parents[1],
    )
    parser.add_argument("--data-dir", type=Path, default=None)
    parser.add_argument("--selection-seed", type=int, default=5768)
    parser.add_argument(
        "--final-seeds",
        type=int,
        nargs="+",
        default=list(OFFICIAL_SEEDS),
    )
    parser.add_argument("--device", default="auto")
    return parser.parse_args()


def candidates() -> list[dict[str, Any]]:
    """Keep the task protocol fixed and vary one architecture factor at a time."""

    base: dict[str, Any] = {
        "max_vocab_size": 10_000,
        "max_length": 200,
        "embedding_dim": 100,
        "hidden_size": 128,
        "dropout": 0.3,
        "lstm_layers": 1,
        "tcn_channels": 64,
        "tcn_kernel_size": 3,
        "tcn_layers": 2,
        "pooling": "mean",
        "learning_rate": 1e-3,
        "batch_size": 32,
        "weight_decay": 1e-4,
        "scheduler_patience": 1,
        "max_epochs": 20,
        "early_stopping_patience": 3,
        "optimizer": "adamw",
        "use_scheduler": True,
        "gradient_clip_norm": 5.0,
        "class_weights": None,
    }
    changes = [
        ("control_hybrid", {}),
        ("mean_max_pool", {"pooling": "mean_max"}),
        ("max_pool", {"pooling": "max"}),
        ("tcn_channels128", {"tcn_channels": 128}),
        ("kernel5", {"tcn_kernel_size": 5}),
        ("tcn_layers3", {"tcn_layers": 3}),
        ("two_layer_lstm", {"lstm_layers": 2}),
        ("dropout50", {"dropout": 0.5}),
        ("embedding200", {"embedding_dim": 200}),
        ("longer_training", {"max_epochs": 30, "early_stopping_patience": 5}),
    ]

    result: list[dict[str, Any]] = []
    for name, overrides in changes:
        config = dict(base)
        config.update(overrides)
        config["candidate"] = name
        result.append(config)
    return result


def write_rows(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fields: list[str] = []
    for row in rows:
        for field in row:
            if field not in fields:
                fields.append(field)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def run_one_hybrid(
    seed: int,
    data_dir: Path,
    split_dir: Path,
    output_dir: Path,
    config: dict[str, Any],
    device: torch.device,
    save_artifacts: bool = True,
    evaluate_test: bool = True,
) -> dict[str, Any]:
    # Required names are defined earlier in this notebook.
    set_seed(seed)
    train_records, test_records = load_seed(data_dir, seed)
    split_train, val_records = create_or_load_validation_split(
        train_records,
        seed,
        split_dir,
    )
    audit = write_overlap_audit(
        train_records,
        test_records,
        seed,
        split_dir.parent / "audits",
    )
    vocabulary = build_vocabulary(split_train, config["max_vocab_size"])
    configured_max_length = config.get("max_length")
    if configured_max_length in (None, "train_max", "longest_train"):
        max_length = max(len(tokenize(record.sentence)) for record in split_train)
    else:
        max_length = int(configured_max_length)

    train_dataset = TextDataset(split_train, vocabulary, max_length)
    val_dataset = TextDataset(val_records, vocabulary, max_length)
    test_dataset = TextDataset(test_records, vocabulary, max_length)
    train_loader = make_loader(train_dataset, config["batch_size"], True, seed)
    val_loader = make_loader(val_dataset, config["batch_size"], False, seed)
    test_loader = make_loader(test_dataset, config["batch_size"], False, seed)
    model = LSTMTCNClassifier(
        vocab_size=len(vocabulary),
        embedding_dim=config["embedding_dim"],
        hidden_size=config["hidden_size"],
        dropout=config["dropout"],
        lstm_layers=config["lstm_layers"],
        tcn_channels=config["tcn_channels"],
        tcn_kernel_size=config["tcn_kernel_size"],
        tcn_layers=config["tcn_layers"],
        pooling=config["pooling"],
        # add
        normalization=config.get("normalization", "none"),
        #done
    )
    training_started = time.perf_counter()
    model, history, best = train_model(
        model,
        train_loader,
        val_loader,
        device,
        config["learning_rate"],
        config["weight_decay"],
        config["scheduler_patience"],
        config["max_epochs"],
        config["early_stopping_patience"],
        optimizer_name=config.get("optimizer", "adamw"),
        use_scheduler=config.get("use_scheduler", True),
        class_weights=config.get("class_weights"),
        gradient_clip_norm=config.get("gradient_clip_norm", 5.0),
    )
    training_seconds = time.perf_counter() - training_started
    criterion = nn.CrossEntropyLoss()
    inference_started = time.perf_counter()
    validation = evaluate_model(model, val_loader, device, criterion)
    test = evaluate_model(model, test_loader, device, criterion) if evaluate_test else None
    inference_seconds = time.perf_counter() - inference_started
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    result = {
        "model": "lstm_tcn",
        "seed": seed,
        "validation": {
            "loss": validation["loss"],
            "accuracy": validation["accuracy"],
            "weighted_f1": validation["weighted_f1"],
            "macro_f1": validation["macro_f1"],
            "per_class_f1": validation["per_class_f1"],
        },
        "test": None if test is None else {
            "loss": test["loss"],
            "accuracy": test["accuracy"],
            "weighted_f1": test["weighted_f1"],
            "macro_f1": test["macro_f1"],
            "per_class_f1": test["per_class_f1"],
        },
        "best_epoch": best["best_epoch"],
        "best_val_weighted_f1": best["best_val_weighted_f1"],
        "train_rows": len(split_train),
        "validation_rows": len(val_records),
        "test_rows": len(test_records),
        "vocabulary_size": len(vocabulary),
        "configured_max_length": configured_max_length,
        "resolved_max_length": max_length,
        "train_max_tokens": max(len(tokenize(record.sentence)) for record in split_train),
        "test_truncated_rows": sum(
            len(tokenize(record.sentence)) > max_length for record in test_records
        ),
        "parameter_count": parameter_count,
        "training_seconds": training_seconds,
        "inference_seconds": inference_seconds,
        "overlap_audit": audit,
    }
    if not save_artifacts:
        return result

    artifact_dir = output_dir / "lstm_tcn" / f"seed_{seed}"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    run_config = {
        **config,
        "model": "lstm_tcn",
        "model_name": "lstm_tcn",
        "seed": seed,
        "random_seed": seed,
        "epochs": config["max_epochs"],
        "configured_max_length": configured_max_length,
        "resolved_max_length": max_length,
        "best_epoch": best["best_epoch"],
        "best_val_weighted_f1": best["best_val_weighted_f1"],
        "data_dir": str(data_dir),
        "split_dir": str(split_dir),
        "output_dir": str(artifact_dir),
        "device": str(device),
        "vocabulary": vocabulary.to_json(),
        "label_names": LABEL_NAMES,
        "checkpoint_revision": None,
        "training_type": "random_embedding",
        "tokenizer": "word-level-regex",
        "train_split_file": str(split_dir / f"{seed}_train.csv"),
        "validation_split_file": str(split_dir / f"{seed}_val.csv"),
        "test_split_file": str(data_dir / f"lab-manual-combine-test-{seed}.xlsx"),
        "library_versions": {
            "python": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
            "numpy": np.__version__,
            "torch": torch.__version__,
        },
    }
    save_json(artifact_dir / "config.json", run_config)
    save_json(artifact_dir / "metrics.json", result)
    write_rows(
        artifact_dir / "training_history.csv",
        history,
    )
    if test is not None:
        save_classification_report(
            artifact_dir / "classification_report.csv",
            test["labels"],
            test["predictions"],
        )
        save_confusion_matrices(artifact_dir, test["labels"], test["predictions"])
        save_predictions_and_errors(artifact_dir, test_records, test, "lstm_tcn", seed)
    save_learning_curve(
        artifact_dir / "learning_curve.png",
        history,
        f"LSTM+TCN seed {seed}",
        best["best_epoch"],
    )
    torch.save(model.state_dict(), artifact_dir / "best_model.pt")
    return result

# add
#!/usr/bin/env python3
"""Second-stage controlled tuning for LSTM, BiLSTM, and LSTM+TCN.

Only seed 5768 validation scores select configurations. The selected
configuration for each path is frozen before the three official test seeds are
evaluated. Existing baseline and earlier experiment directories are never
overwritten.
"""


import argparse
import csv
import json
import os
import statistics
import sys
import time
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import torch


OFFICIAL_SEEDS = (5768, 78516, 944601)
MODEL_ORDER = ("lstm", "bilstm", "lstm_tcn")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--repo-root",
        type=Path,
        default=Path(__file__).resolve().parents[1],
    )
    parser.add_argument("--data-dir", type=Path, default=None)
    parser.add_argument("--selection-seed", type=int, default=5768)
    parser.add_argument("--final-seeds", type=int, nargs="+", default=list(OFFICIAL_SEEDS))
    parser.add_argument("--device", default="auto")
    parser.add_argument("--glove-path", type=Path, default=None)
    return parser.parse_args()


def base_rnn_config() -> dict[str, Any]:
    return {
        "max_vocab_size": 10_000,
        "max_length": 200,
        "embedding_dim": 100,
        "hidden_size": 128,
        "dropout": 0.3,
        "learning_rate": 1e-3,
        "batch_size": 32,
        "weight_decay": 1e-4,
        "scheduler_patience": 1,
        "max_epochs": 20,
        "early_stopping_patience": 3,
        "optimizer": "adamw",
        "use_scheduler": True,
        "gradient_clip_norm": 5.0,
        "class_weights": None,
        "pooling": "final_hidden",
        "glove_path": None,
        "freeze_embeddings": True,
        "architecture_variant": "single_layer",
    }


def rnn_candidates(model_name: str, glove_path: Path | None) -> list[dict[str, Any]]:
    """Use interpretable one-factor and focused interaction candidates."""

    changes = [
        ("control", {}),
        ("hidden64", {"hidden_size": 64}),
        ("embedding200", {"embedding_dim": 200}),
        ("dropout20", {"dropout": 0.2}),
        ("dropout40", {"dropout": 0.4}),
        ("dropout50", {"dropout": 0.5}),
        ("lr3e-4", {"learning_rate": 3e-4}),
        ("lr5e-4", {"learning_rate": 5e-4}),
        ("batch16", {"batch_size": 16}),
        ("weight_decay0", {"weight_decay": 0.0}),
        ("weight_decay5e-4", {"weight_decay": 5e-4}),
        ("mean_pool", {"pooling": "mean"}),
        ("max_pool", {"pooling": "max"}),
        ("attention", {"pooling": "attention"}),
        (
            "embedding200_dropout40",
            {"embedding_dim": 200, "dropout": 0.4},
        ),
        (
            "hidden64_dropout40",
            {"hidden_size": 64, "dropout": 0.4},
        ),
        (
            "lr5e-4_dropout40",
            {"learning_rate": 5e-4, "dropout": 0.4},
        ),
        (
            "mean_pool_longer",
            {"pooling": "mean", "max_epochs": 30, "early_stopping_patience": 5},
        ),
    ]
    configs: list[dict[str, Any]] = []
    for candidate, overrides in changes:
        config = base_rnn_config()
        config.update(overrides)
        config["candidate"] = candidate
        config["model"] = model_name
        configs.append(config)

    if model_name == "bilstm" and glove_path is not None:
        for candidate, freeze in (
            ("glove_frozen", True),
            ("glove_trainable", False),
        ):
            config = base_rnn_config()
            config.update(
                {
                    "candidate": candidate,
                    "model": model_name,
                    "embedding_dim": 100,
                    "glove_path": str(glove_path),
                    "freeze_embeddings": freeze,
                    "training_type": "static_pretrained_embedding",
                }
            )
            configs.append(config)
    return configs


def tcn_candidates() -> list[dict[str, Any]]:
    base = {
        "max_vocab_size": 10_000,
        "max_length": 200,
        "embedding_dim": 100,
        "hidden_size": 128,
        "dropout": 0.3,
        "lstm_layers": 1,
        "tcn_channels": 64,
        "tcn_kernel_size": 3,
        "tcn_layers": 2,
        "pooling": "mean",
        "normalization": "none",
        "learning_rate": 1e-3,
        "batch_size": 32,
        "weight_decay": 1e-4,
        "scheduler_patience": 1,
        "max_epochs": 20,
        "early_stopping_patience": 3,
        "optimizer": "adamw",
        "use_scheduler": True,
        "gradient_clip_norm": 5.0,
        "class_weights": None,
        "architecture_variant": "single_layer_lstm_plus_tcn",
    }
    changes = [
        ("control_hybrid", {}),
        ("longer_training", {"max_epochs": 30, "early_stopping_patience": 5}),
        ("mean_max_pool", {"pooling": "mean_max"}),
        ("max_pool", {"pooling": "max"}),
        ("channels32", {"tcn_channels": 32}),
        ("channels64", {"tcn_channels": 64}),
        ("tcn_layers3", {"tcn_layers": 3}),
        ("kernel5", {"tcn_kernel_size": 5}),
        ("dropout20", {"dropout": 0.2}),
        ("dropout40", {"dropout": 0.4}),
        ("lr3e-4", {"learning_rate": 3e-4}),
        ("lr5e-4", {"learning_rate": 5e-4}),
        ("weight_decay5e-4", {"weight_decay": 5e-4}),
        ("channel_norm", {"normalization": "channel"}),
        (
            "longer_mean_max",
            {
                "pooling": "mean_max",
                "max_epochs": 30,
                "early_stopping_patience": 5,
            },
        ),
        (
            "longer_lr5e-4",
            {
                "learning_rate": 5e-4,
                "max_epochs": 30,
                "early_stopping_patience": 5,
            },
        ),
        (
            "longer_channel_norm",
            {
                "normalization": "channel",
                "max_epochs": 30,
                "early_stopping_patience": 5,
            },
        ),
        (
            "longer_mean_max_channel_norm",
            {
                "pooling": "mean_max",
                "normalization": "channel",
                "max_epochs": 30,
                "early_stopping_patience": 5,
            },
        ),
    ]
    configs: list[dict[str, Any]] = []
    for candidate, overrides in changes:
        config = dict(base)
        config.update(overrides)
        config["candidate"] = candidate
        config["model"] = "lstm_tcn"
        configs.append(config)
    return configs


def write_rows(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    rows = list(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields: list[str] = []
    for row in rows:
        for field in row:
            if field not in fields:
                fields.append(field)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def read_csv(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        return []
    with path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def find_glove(repo_root: Path, requested: Path | None) -> tuple[Path | None, dict[str, Any]]:
    if requested is not None:
        requested = requested.expanduser().resolve()
        if requested.exists():
            return requested, {"status": "available", "path": str(requested), "source": "cli"}
        return None, {
            "status": "skipped",
            "reason": "The requested GloVe path does not exist; no external download was attempted.",
            "requested_path": str(requested),
        }

    candidates = [
        repo_root / "data" / "glove.6B.100d.txt",
        repo_root / "data" / "glove.100d.txt",
        repo_root.parent / "glove.6B.100d.txt",
        repo_root.parent / "glove.100d.txt",
    ]
    for path in candidates:
        if path.exists():
            return path, {"status": "available", "path": str(path), "source": "local_discovery"}
    return None, {
        "status": "skipped",
        "reason": "No local 100d GloVe file was found; no external download was attempted.",
        "searched_paths": [str(path) for path in candidates],
    }


def run_candidate(
    model_name: str,
    seed: int,
    data_dir: Path,
    split_dir: Path,
    output_dir: Path,
    config: dict[str, Any],
    device: torch.device,
) -> dict[str, Any]:
    if model_name == "lstm_tcn":
        return run_one_hybrid(
            seed, data_dir, split_dir, output_dir, config, device,
            save_artifacts=False, evaluate_test=False,
        )
    return run_one(
        model_name, seed, data_dir, split_dir, output_dir, config, device,
        save_artifacts=False, evaluate_test=False,
    )


def rank_and_save_search(
    model_name: str,
    configs: list[dict[str, Any]],
    rows: list[dict[str, Any]],
    search_dir: Path,
) -> dict[str, Any]:
    control_name = "control_hybrid" if model_name == "lstm_tcn" else "control"
    control_scores = [
        float(row["validation_weighted_f1"])
        for row in rows
        if row["candidate"] == control_name
    ]
    control_score = max(control_scores) if control_scores else float("-inf")
    ranked = sorted(
        rows,
        key=lambda row: (
            -float(row["validation_weighted_f1"]),
            -float(row["validation_macro_f1"]),
            int(row["parameter_count"]),
        ),
    )
    selected_candidate = ranked[0]["candidate"]
    for row in rows:
        if row["candidate"] == selected_candidate:
            row["attempt_status"] = "selected"
        elif float(row["validation_weighted_f1"]) < control_score - 1e-9:
            row["attempt_status"] = "failed_to_improve_control"
        else:
            row["attempt_status"] = "not_selected"
    config_by_candidate = {config["candidate"]: config for config in configs}
    best_config = dict(config_by_candidate[selected_candidate])
    search_dir.mkdir(parents=True, exist_ok=True)
    write_rows(search_dir / f"{model_name}_validation_search.csv", ranked)
    best_row = next(row for row in ranked if row["candidate"] == selected_candidate)
    with (search_dir / f"best_{model_name}_config.json").open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "model": model_name,
                "selection_seed": 5768,
                "metric": "validation_weighted_f1",
                "config": best_config,
                "validation_weighted_f1": best_row["validation_weighted_f1"],
                "validation_macro_f1": best_row["validation_macro_f1"],
                "parameter_count": best_row["parameter_count"],
                "attempt_status": "selected",
            },
            handle,
            indent=2,
            ensure_ascii=False,
        )
    return best_config


def aggregate_result_dicts(results: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    aggregate: dict[str, dict[str, Any]] = {}
    for model_name in MODEL_ORDER:
        model_results = [result for result in results if result["model"] == model_name]
        if not model_results:
            continue
        metrics = {
            "test_weighted_f1": [result["test"]["weighted_f1"] for result in model_results],
            "test_macro_f1": [result["test"]["macro_f1"] for result in model_results],
            "test_accuracy": [result["test"]["accuracy"] for result in model_results],
            "dovish_f1": [result["test"]["per_class_f1"][0] for result in model_results],
            "hawkish_f1": [result["test"]["per_class_f1"][1] for result in model_results],
            "neutral_f1": [result["test"]["per_class_f1"][2] for result in model_results],
        }
        aggregate[model_name] = {
            metric: {
                "mean": statistics.mean(values),
                "std": statistics.stdev(values) if len(values) > 1 else 0.0,
                "values": values,
            }
            for metric, values in metrics.items()
        }
        aggregate[model_name]["parameter_count_mean"] = statistics.mean(
            result["parameter_count"] for result in model_results
        )
        aggregate[model_name]["mean_training_seconds"] = statistics.mean(
            result["training_seconds"] for result in model_results
        )
    return aggregate


def write_comparison(
    output_root: Path,
    final_results: list[dict[str, Any]],
    baseline_path: Path,
) -> dict[str, Any]:
    aggregate = aggregate_result_dicts(final_results)
    baseline_rows = read_csv(baseline_path)
    baseline_values: dict[str, dict[str, list[float]]] = {}
    for row in baseline_rows:
        model = row["model"]
        baseline_values.setdefault(model, {})
        for metric in ("test_weighted_f1", "test_macro_f1", "test_accuracy"):
            baseline_values[model].setdefault(metric, []).append(float(row[metric]))
    for model, values in baseline_values.items():
        values["test_weighted_f1_mean"] = [statistics.mean(values["test_weighted_f1"])]
        values["test_macro_f1_mean"] = [statistics.mean(values["test_macro_f1"])]
        values["test_accuracy_mean"] = [statistics.mean(values["test_accuracy"])]

    rows: list[dict[str, Any]] = []
    for model_name in MODEL_ORDER:
        if model_name not in aggregate:
            continue
        metric = aggregate[model_name]
        references = ["lstm", "bilstm"] if model_name == "lstm_tcn" else [model_name]
        for reference in references:
            reference_mean = baseline_values[reference]["test_weighted_f1_mean"][0]
            values = metric["test_weighted_f1"]["values"]
            baseline_seed_values = baseline_values[reference]["test_weighted_f1"]
            rows.append(
                {
                    "model": model_name,
                    "baseline_reference": reference,
                    "weighted_f1_mean": metric["test_weighted_f1"]["mean"],
                    "weighted_f1_std": metric["test_weighted_f1"]["std"],
                    "macro_f1_mean": metric["test_macro_f1"]["mean"],
                    "macro_f1_std": metric["test_macro_f1"]["std"],
                    "accuracy_mean": metric["test_accuracy"]["mean"],
                    "dovish_f1_mean": metric["dovish_f1"]["mean"],
                    "hawkish_f1_mean": metric["hawkish_f1"]["mean"],
                    "neutral_f1_mean": metric["neutral_f1"]["mean"],
                    "parameter_count_mean": metric["parameter_count_mean"],
                    "mean_training_seconds": metric["mean_training_seconds"],
                    "weighted_f1_delta_vs_baseline": metric["test_weighted_f1"]["mean"] - reference_mean,
                    "wins_vs_baseline": sum(
                        value > baseline
                        for value, baseline in zip(values, baseline_seed_values)
                    ),
                    "status": "mean_improved"
                    if metric["test_weighted_f1"]["mean"] > reference_mean
                    else "not_improved",
                }
            )
    write_rows(output_root / "three_path_comparison.csv", rows)
    comparison_payload = {
        "primary_metric": "test_weighted_f1_mean",
        "baseline_summary": str(baseline_path),
        "rows": rows,
        "aggregate": aggregate,
    }
    with (output_root / "three_path_comparison.json").open("w", encoding="utf-8") as handle:
        json.dump(comparison_payload, handle, indent=2, ensure_ascii=False)
    return comparison_payload


def save_unified_figures(output_root: Path, aggregate: dict[str, dict[str, Any]]) -> None:
    """Write the required three-path metric and cost figures."""

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    labels = ["LSTM", "BiLSTM", "LSTM+TCN"]
    model_names = ["lstm", "bilstm", "lstm_tcn"]
    colours = ["#4c78a8", "#f58518", "#54a24b"]

    def metric_plot(metric: str, filename: str, title: str, ylabel: str) -> None:
        means = [aggregate[name][metric]["mean"] for name in model_names]
        stds = [aggregate[name][metric]["std"] for name in model_names]
        figure, axis = plt.subplots(figsize=(8, 5))
        axis.bar(labels, means, yerr=stds, capsize=5, color=colours)
        axis.set_ylim(0.0, 1.0)
        axis.set_ylabel(ylabel)
        axis.set_title(title)
        axis.grid(axis="y", alpha=0.25)
        for index, value in enumerate(means):
            axis.text(index, value + stds[index] + 0.015, f"{value:.3f}", ha="center")
        figure.tight_layout()
        figure.savefig(output_root / filename, dpi=180)
        plt.close(figure)

    metric_plot(
        "test_weighted_f1",
        "three_paths_weighted_f1.png",
        "LSTM, BiLSTM, and LSTM+TCN weighted F1",
        "Test weighted F1 mean +/- std",
    )
    metric_plot(
        "test_macro_f1",
        "three_paths_macro_f1.png",
        "LSTM, BiLSTM, and LSTM+TCN macro F1",
        "Test macro F1 mean +/- std",
    )

    per_class = {
        "Dovish": "dovish_f1",
        "Hawkish": "hawkish_f1",
        "Neutral": "neutral_f1",
    }
    figure, axis = plt.subplots(figsize=(8, 5))
    x = np.arange(len(labels))
    width = 0.24
    for offset, (class_name, metric) in enumerate(per_class.items()):
        axis.bar(
            x + (offset - 1) * width,
            [aggregate[name][metric]["mean"] for name in model_names],
            width,
            label=class_name,
        )
    axis.set_xticks(x, labels)
    axis.set_ylim(0.0, 1.0)
    axis.set_ylabel("Test F1 mean")
    axis.set_title("Per-class F1 comparison")
    axis.legend()
    axis.grid(axis="y", alpha=0.25)
    figure.tight_layout()
    figure.savefig(output_root / "three_paths_per_class_f1.png", dpi=180)
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(8, 5))
    parameters = [aggregate[name]["parameter_count_mean"] for name in model_names]
    training_seconds = [aggregate[name]["mean_training_seconds"] for name in model_names]
    axis.scatter(parameters, training_seconds, c=colours, s=100)
    for label, parameter_count, seconds in zip(labels, parameters, training_seconds):
        axis.annotate(label, (parameter_count, seconds), xytext=(6, 6), textcoords="offset points")
    axis.set_xlabel("Mean parameter count")
    axis.set_ylabel("Mean training time (seconds)")
    axis.set_title("Performance cost inputs: parameter count and training time")
    axis.grid(alpha=0.25)
    figure.tight_layout()
    figure.savefig(output_root / "three_paths_cost.png", dpi=180)
    plt.close(figure)

def run_embedded_baseline(data_dir, split_dir, output_root, device, seeds=OFFICIAL_SEEDS):
    config = {
        "max_vocab_size": 10000, "max_length": 200, "embedding_dim": 100,
        "hidden_size": 128, "dropout": 0.3, "learning_rate": 1e-3,
        "batch_size": 32, "weight_decay": 1e-4, "scheduler_patience": 1,
        "max_epochs": 20, "early_stopping_patience": 3, "optimizer": "adamw",
        "use_scheduler": True, "gradient_clip_norm": 5.0, "class_weights": None,
        "pooling": "final_hidden",
    }
    results = []
    for model_name in ("lstm", "bilstm"):
        for seed in seeds:
            results.append(run_one(
                model_name, seed, data_dir, split_dir, output_root, dict(config),
                device, save_artifacts=True, evaluate_test=True,
            ))
    save_summary(output_root, results)
    return results


def run_embedded_second_stage(repo_root, data_dir, split_dir, output_root,
                              device, final_seeds=OFFICIAL_SEEDS):
    search_dir = output_root / "validation_search"
    final_dir = output_root / "final"
    glove_path, glove_status = find_glove(repo_root, None)
    save_json(output_root / "glove_status.json", glove_status)
    candidate_sets = {
        "lstm": rnn_candidates("lstm", None),
        "bilstm": rnn_candidates("bilstm", glove_path),
        "lstm_tcn": tcn_candidates(),
    }
    best_configs, all_rows = {}, []
    for model_name in MODEL_ORDER:
        configs, rows = candidate_sets[model_name], []
        for number, config in enumerate(configs, 1):
            result = run_candidate(
                model_name, 5768, data_dir, split_dir, search_dir / "runs",
                dict(config), device,
            )
            rows.append({
                **config, "candidate_number": number,
                "validation_weighted_f1": result["validation"]["weighted_f1"],
                "validation_macro_f1": result["validation"]["macro_f1"],
                "validation_accuracy": result["validation"]["accuracy"],
                "parameter_count": result["parameter_count"],
                "best_epoch": result["best_epoch"],
            })
        best_configs[model_name] = rank_and_save_search(
            model_name, configs, rows, search_dir,
        )
        all_rows.extend(rows)
    write_rows(search_dir / "three_paths_validation_search.csv", all_rows)
    save_json(search_dir / "selected_configs.json", {
        "selection_seed": 5768,
        "metric": "validation_weighted_f1",
        "tie_breaks": ["validation_macro_f1", "parameter_count"],
        "configs": best_configs,
    })

    final_results = []
    for model_name in MODEL_ORDER:
        for seed in final_seeds:
            config = best_configs[model_name]
            if model_name == "lstm_tcn":
                result = run_one_hybrid(
                    seed, data_dir, split_dir, final_dir, config, device,
                    save_artifacts=True, evaluate_test=True,
                )
            else:
                result = run_one(
                    model_name, seed, data_dir, split_dir, final_dir, config, device,
                    save_artifacts=True, evaluate_test=True,
                )
            final_results.append(result)
    save_summary(final_dir, final_results)
    comparison = write_comparison(
        output_root, final_results,
        repo_root / "results/baseline_unsw/lstm_bilstm_summary.csv",
    )
    save_unified_figures(output_root, comparison["aggregate"])
    save_json(output_root / "run_metadata.json", {
        "selection_seed": 5768,
        "final_seeds": list(final_seeds),
        "device": str(device),
        "best_configs": best_configs,
        "glove_status": glove_status,
        "notebook_runner": "embedded_self_contained_runner",
    })
    return comparison


def run_embedded_current_experiments(repo_root, data_dir, split_dir, device):
    baseline_marker = repo_root / "results/baseline_unsw/lstm_bilstm_summary.csv"
    v2_marker = repo_root / "results/lstm_three_paths_v2/three_path_comparison.csv"
    if not baseline_marker.exists():
        run_embedded_baseline(
            data_dir, split_dir, repo_root / "results/baseline_unsw", device,
        )
    if not v2_marker.exists():
        run_embedded_second_stage(
            repo_root, data_dir, split_dir,
            repo_root / "results/lstm_three_paths_v2", device,
        )

#done


In [ ]:
CACHED_MESSAGE = '所有历史结果文件都已存在，Notebook 将直接展示缓存结果。'
MISSING_MESSAGE = '历史结果缓存不完整；Notebook 只会使用上方嵌入的 runner，不会调用外部脚本。'
LABEL_MESSAGE = '标签：'
from pathlib import Path
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display


def locate_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the COMP9444_FOMC_Analysis repository.")


REPO_ROOT = locate_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
DATA_DIR = resolve_data_dir(REPO_ROOT.parent / "FOMC_dataset_checkpoint")
SPLIT_DIR = REPO_ROOT / "data" / "splits"
SEEDS = (5768, 78516, 944601)
DEVICE = choose_device("auto")
AUTO_RUN_MISSING = True

RESULT_ROOTS = {
    "baseline": REPO_ROOT / "results" / "baseline_unsw",
    "bilstm_tuning": REPO_ROOT / "results" / "lstm_bilstm_tuning_v1",
    "lstm_v2": REPO_ROOT / "results" / "lstm_v2",
    "lstm_v3": REPO_ROOT / "results" / "lstm_v3",
    "lstm_tcn": REPO_ROOT / "results" / "lstm_tcn_tuning_v1",
    "lstm_three_paths_v2": REPO_ROOT / "results" / "lstm_three_paths_v2",
}
SUMMARY_RELATIVE = Path("final") / "lstm_bilstm_summary.csv"

JOBS = [
    ("same-environment baseline", RESULT_ROOTS["baseline"] / "lstm_bilstm_summary.csv"),
    ("LSTM/BiLSTM tuning", RESULT_ROOTS["bilstm_tuning"] / SUMMARY_RELATIVE),
    ("LSTM second-round attempt", RESULT_ROOTS["lstm_v2"] / SUMMARY_RELATIVE),
    ("LSTM third-round attempt", RESULT_ROOTS["lstm_v3"] / SUMMARY_RELATIVE),
    ("LSTM+TCN hybrid search", RESULT_ROOTS["lstm_tcn"] / SUMMARY_RELATIVE),
    ("Second-stage v2 search", RESULT_ROOTS["lstm_three_paths_v2"] / "three_path_comparison.csv"),
]


def ensure_experiment_outputs():
    required_baseline = RESULT_ROOTS["baseline"] / "lstm_bilstm_summary.csv"
    required_v2 = RESULT_ROOTS["lstm_three_paths_v2"] / "three_path_comparison.csv"
    if AUTO_RUN_MISSING and (not required_baseline.exists() or not required_v2.exists()):
        run_embedded_current_experiments(REPO_ROOT, DATA_DIR, SPLIT_DIR, DEVICE)
    missing = [label for label, marker in JOBS if not marker.exists()]
    if missing:
        print(MISSING_MESSAGE)
    else:
        print(CACHED_MESSAGE)


ensure_experiment_outputs()
print("Repository:", REPO_ROOT)
print("Data:", DATA_DIR)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
display(Markdown(LABEL_MESSAGE + ", ".join(f"{key} = {value}" for key, value in LABEL_NAMES.items())))


In [ ]:

dataset_rows = []
audit_rows = []
for seed in SEEDS:
    source_train, source_test = load_seed(DATA_DIR, seed)
    train_part, validation_part = create_or_load_validation_split(
        source_train, seed, SPLIT_DIR
    )
    audit = write_overlap_audit(
        source_train, source_test, seed, SPLIT_DIR.parent / "audits"
    )
    dataset_rows.append(
        {
            "seed": seed,
            "source_train_rows": len(source_train),
            "effective_train_rows": len(train_part),
            "validation_rows": len(validation_part),
            "test_rows": len(source_test),
            "max_train_tokens": max(len(tokenize(record.sentence)) for record in train_part),
        }
    )
    audit_rows.append(
        {
            "seed": seed,
            "index_overlap": audit["index_overlap_count"],
            "normalised_text_overlap": audit["normalised_sentence_overlap_count"],
            "duplicate_train_sentences": audit["duplicate_sentences_within_train"],
            "duplicate_test_sentences": audit["duplicate_sentences_within_test"],
        }
    )

display(Markdown("## Dataset and split audit"))
display(pd.DataFrame(dataset_rows))
display(pd.DataFrame(audit_rows))


## 调参历史，包括失败尝试

下面的搜索表会保留所有候选。分数较低的候选不会删除，而是标记为低于 control 或没有被选中。这样 Notebook 能够展示最终配置为什么被选中，而不是只展示成功结果。

In [ ]:

SEARCH_SOURCES = [
    (
        "paper reconstruction search",
        REPO_ROOT / "results" / "lstm_bilstm_paper_optimisation" / "paper_search",
    ),
    (
        "legacy optimisation search",
        REPO_ROOT / "results" / "lstm_bilstm_paper_optimisation" / "optimisation_search",
    ),
    (
        "LSTM/BiLSTM tuning v1",
        RESULT_ROOTS["bilstm_tuning"] / "validation_search",
    ),
    (
        "LSTM tuning v2",
        RESULT_ROOTS["lstm_v2"] / "validation_search",
    ),
    (
        "LSTM tuning v3",
        RESULT_ROOTS["lstm_v3"] / "validation_search",
    ),
    (
        "LSTM+TCN tuning",
        RESULT_ROOTS["lstm_tcn"] / "validation_search",
    ),
]

history_frames = []
for group_name, directory in SEARCH_SOURCES:
    for path in sorted(directory.glob("*validation_search.csv")):
        frame = pd.read_csv(path)
        if "validation_weighted_f1" not in frame.columns:
            frame["validation_weighted_f1"] = frame["validation_weighted_f1_mean"]
        if "validation_macro_f1" not in frame.columns:
            frame["validation_macro_f1"] = frame["validation_macro_f1_mean"]
        if "candidate" not in frame.columns:
            frame["candidate"] = frame.get("run", pd.Series(range(1, len(frame) + 1)))
        frame["attempt_group"] = group_name
        frame["search_file"] = str(path.relative_to(REPO_ROOT))
        history_frames.append(frame)

history = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()
if history.empty:
    display(Markdown("No validation search CSV files were found."))
else:
    status_frames = []
    for group_name, group in history.groupby("attempt_group", sort=False):
        group = group.copy()
        best_index = group["validation_weighted_f1"].idxmax()
        control_rows = group[
            group["candidate"].astype(str).str.contains("control|legacy", case=False, regex=True)
        ]
        control_score = (
            control_rows["validation_weighted_f1"].max()
            if not control_rows.empty
            else group["validation_weighted_f1"].min()
        )
        group["attempt_status"] = np.select(
            [
                group.index == best_index,
                group["validation_weighted_f1"] < control_score - 1e-9,
            ],
            ["selected", "failed_to_improve_control"],
            default="not_selected",
        )
        status_frames.append(group)
    history = pd.concat(status_frames, ignore_index=True)
    history_view = history[
        [
            "attempt_group",
            "model",
            "candidate",
            "attempt_status",
            "validation_weighted_f1",
            "validation_macro_f1",
            "parameter_count",
        ]
    ].sort_values(
        ["attempt_group", "model", "validation_weighted_f1"],
        ascending=[True, True, False],
    )
    display(Markdown(
        "## Visible tuning history\n\n"
        "The failed or not-selected candidates are intentionally kept. "
        "A candidate is labelled failed when its validation weighted F1 is below "
        "that search's control configuration; test data was not used for this label."
    ))
    display(history_view.reset_index(drop=True))


In [ ]:

def read_summary(root):
    paths = [
        root / "lstm_bilstm_summary.csv",
        root / SUMMARY_RELATIVE,
    ]
    for path in paths:
        if path.exists():
            return pd.read_csv(path)
    return pd.DataFrame()

baseline_summary = read_summary(RESULT_ROOTS["baseline"])
tuned_summary = read_summary(RESULT_ROOTS["bilstm_tuning"])
v2_summary = read_summary(RESULT_ROOTS["lstm_v2"])
v3_summary = read_summary(RESULT_ROOTS["lstm_v3"])
tcn_summary = read_summary(RESULT_ROOTS["lstm_tcn"])
legacy_paper_root = REPO_ROOT / "results" / "lstm_bilstm_paper_optimisation"
paper_summary = read_summary(legacy_paper_root / "paper_replication")
legacy_optimised_summary = read_summary(legacy_paper_root / "optimised")

def aggregate_rows(summary, experiment_name, model_name, protocol_note):
    group = summary[summary["model"] == model_name].copy()
    if group.empty:
        return []
    return [
        {
            "experiment": experiment_name,
            "model": model_name,
            "protocol": protocol_note,
            "weighted_f1_mean": group["test_weighted_f1"].mean(),
            "weighted_f1_std": group["test_weighted_f1"].std(),
            "macro_f1_mean": group["test_macro_f1"].mean(),
            "macro_f1_std": group["test_macro_f1"].std(),
            "accuracy_mean": group["test_accuracy"].mean(),
            "parameter_count_mean": group["parameter_count"].mean(),
        }
    ]

comparison_rows = []
comparison_rows += aggregate_rows(
    baseline_summary, "same-environment baseline", "lstm", "current baseline"
)
comparison_rows += aggregate_rows(
    baseline_summary, "same-environment baseline", "bilstm", "current baseline"
)
comparison_rows += aggregate_rows(
    tuned_summary, "tuned LSTM", "lstm", "validation-selected LSTM"
)
comparison_rows += aggregate_rows(
    tuned_summary, "tuned BiLSTM", "bilstm", "validation-selected BiLSTM"
)
comparison_rows += aggregate_rows(
    v2_summary, "LSTM v2", "lstm", "failed improvement attempt"
)
comparison_rows += aggregate_rows(
    v3_summary, "LSTM v3", "lstm", "no-gain control attempt"
)
comparison_rows += aggregate_rows(
    tcn_summary, "LSTM+TCN", "lstm_tcn", "additional hybrid architecture ablation"
)
comparison_rows += aggregate_rows(
    paper_summary, "paper reconstruction (legacy)", "lstm", "different paper-matched protocol"
)
comparison_rows += aggregate_rows(
    paper_summary, "paper reconstruction (legacy)", "bilstm", "different paper-matched protocol"
)
comparison_rows += aggregate_rows(
    legacy_optimised_summary, "legacy paper optimisation", "lstm", "different legacy protocol"
)
comparison_rows += aggregate_rows(
    legacy_optimised_summary, "legacy paper optimisation", "bilstm", "different legacy protocol"
)

comparison = pd.DataFrame(comparison_rows)
display(Markdown("## Final three-seed comparison"))
display(comparison.round(4))


In [ ]:

def seed_metrics(summary, label, model_name):
    frame = summary[summary["model"] == model_name][
        ["seed", "test_weighted_f1", "test_macro_f1", "test_accuracy"]
    ].copy()
    return frame.rename(
        columns={
            "test_weighted_f1": f"{label} weighted F1",
            "test_macro_f1": f"{label} macro F1",
            "test_accuracy": f"{label} accuracy",
        }
    )

seed_comparison = pd.DataFrame({"seed": list(SEEDS)})
for frame in (
    seed_metrics(baseline_summary, "LSTM baseline", "lstm"),
    seed_metrics(tuned_summary, "BiLSTM tuned", "bilstm"),
    seed_metrics(tcn_summary, "LSTM+TCN", "lstm_tcn"),
):
    seed_comparison = seed_comparison.merge(frame, on="seed", how="left")

seed_comparison["TCN weighted F1 minus LSTM"] = (
    seed_comparison["LSTM+TCN weighted F1"]
    - seed_comparison["LSTM baseline weighted F1"]
)
seed_comparison["TCN weighted F1 minus BiLSTM"] = (
    seed_comparison["LSTM+TCN weighted F1"]
    - seed_comparison["BiLSTM tuned weighted F1"]
)
display(Markdown("## Per-seed comparison"))
display(seed_comparison.round(4))


In [ ]:

TCN_OUTPUT = RESULT_ROOTS["lstm_tcn"] / "validation_search"
tcn_config = json.loads((TCN_OUTPUT / "best_config.json").read_text())
display(Markdown("## Selected LSTM+TCN configuration"))
display(pd.DataFrame([tcn_config["config"]]).T.rename(columns={0: "value"}))


torch.manual_seed(9444)
smoke_model = LSTMTCNClassifier(
    vocab_size=128,
    embedding_dim=16,
    hidden_size=12,
    dropout=0.3,
    lstm_layers=2,
    tcn_channels=10,
    tcn_kernel_size=3,
    tcn_layers=3,
    pooling="mean_max",
)
smoke_ids = torch.randint(0, 128, (4, 20))
smoke_lengths = torch.tensor([20, 15, 8, 1])
smoke_logits = smoke_model(smoke_ids, smoke_lengths)
assert tuple(smoke_logits.shape) == (4, 3)
display(pd.DataFrame([{
    "test": "LSTM+TCN forward-pass smoke test",
    "input_shape": tuple(smoke_ids.shape),
    "output_shape": tuple(smoke_logits.shape),
    "multi_layer_configuration": True,
    "parameter_count": sum(parameter.numel() for parameter in smoke_model.parameters()),
    "status": "PASS",
}]))


In [ ]:

display(Markdown("## Validation search curve"))
if not history.empty:
    tcn_history = history[history["attempt_group"] == "LSTM+TCN tuning"].copy()
    if not tcn_history.empty:
        tcn_history["label"] = tcn_history["candidate"].astype(str)
        tcn_history = tcn_history.sort_values("validation_weighted_f1", ascending=True)
        figure, axis = plt.subplots(figsize=(10, 5))
        axis.barh(tcn_history["label"], tcn_history["validation_weighted_f1"], color="#2f6f8f")
        axis.set_xlabel("Seed 5768 validation weighted F1")
        axis.set_title("LSTM+TCN candidate history, including non-selected attempts")
        axis.grid(axis="x", alpha=0.25)
        figure.tight_layout()
        plt.show()

def show_image(path):
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("Missing image:", path)

for label, root in (
    ("Same-environment baseline", RESULT_ROOTS["baseline"]),
    ("Tuned LSTM/BiLSTM", RESULT_ROOTS["bilstm_tuning"] / "final"),
    ("LSTM+TCN", RESULT_ROOTS["lstm_tcn"] / "final"),
):
    display(Markdown(f"### {label} aggregate plots"))
    show_image(root / "lstm_bilstm_weighted_f1.png")
    show_image(root / "lstm_bilstm_macro_f1.png")


In [ ]:

def show_run_artifacts(title, root, model_name, seed):
    run_dir = root / model_name / f"seed_{seed}"
    display(Markdown(f"## {title}: {model_name}, seed {seed}"))
    metrics_path = run_dir / "metrics.json"
    report_path = run_dir / "classification_report.csv"
    matrix_path = run_dir / "confusion_matrix_normalized.png"
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text())
        display(pd.DataFrame([metrics["test"]]))
    if report_path.exists():
        display(pd.read_csv(report_path))
    show_image(matrix_path)

best_tcn_seed = int(
    tcn_summary.sort_values("test_weighted_f1", ascending=False).iloc[0]["seed"]
)
best_bilstm_seed = int(
    tuned_summary[tuned_summary["model"] == "bilstm"]
    .sort_values("test_weighted_f1", ascending=False)
    .iloc[0]["seed"]
)
for seed in SEEDS:
    show_run_artifacts(
        "LSTM+TCN confusion matrix and report",
        RESULT_ROOTS["lstm_tcn"] / "final",
        "lstm_tcn",
        seed,
    )
show_run_artifacts(
    "Best tuned BiLSTM confusion matrix and report",
    RESULT_ROOTS["bilstm_tuning"] / "final",
    "bilstm",
    best_bilstm_seed,
)


In [ ]:

for title, root, model_name, seed in (
    (
        "LSTM+TCN learning curve and errors",
        RESULT_ROOTS["lstm_tcn"] / "final",
        "lstm_tcn",
        best_tcn_seed,
    ),
    (
        "Best tuned BiLSTM learning curve and errors",
        RESULT_ROOTS["bilstm_tuning"] / "final",
        "bilstm",
        best_bilstm_seed,
    ),
):
    run_dir = root / model_name / f"seed_{seed}"
    display(Markdown(f"## {title}"))
    show_image(run_dir / "learning_curve.png")
    error_path = run_dir / "selected_error_cases.csv"
    if error_path.exists():
        display(pd.read_csv(error_path)[
            ["true_label_name", "predicted_label_name", "category", "sentence"]
        ])


In [ ]:

tcn_row = comparison[comparison["model"] == "lstm_tcn"].iloc[0]
lstm_row = comparison[
    (comparison["experiment"] == "same-environment baseline")
    & (comparison["model"] == "lstm")
].iloc[0]
bilstm_row = comparison[
    (comparison["experiment"] == "tuned BiLSTM")
    & (comparison["model"] == "bilstm")
].iloc[0]
tcn_wins = int(
    (
        seed_comparison["LSTM+TCN weighted F1"]
        > seed_comparison[["LSTM baseline weighted F1", "BiLSTM tuned weighted F1"]].max(axis=1)
    ).sum()
)
display(Markdown(
    "## Conclusions\n\n"
    f"The selected LSTM+TCN reaches weighted F1 "
    f"{tcn_row['weighted_f1_mean']:.4f} +/- {tcn_row['weighted_f1_std']:.4f} "
    f"and macro F1 {tcn_row['macro_f1_mean']:.4f} +/- {tcn_row['macro_f1_std']:.4f}. "
    f"It improves the three-seed weighted-F1 mean over the same-environment LSTM "
    f"by {tcn_row['weighted_f1_mean'] - lstm_row['weighted_f1_mean']:.4f} "
    f"and over the tuned BiLSTM by {tcn_row['weighted_f1_mean'] - bilstm_row['weighted_f1_mean']:.4f}. "
    f"It beats both reference models on {tcn_wins} of the {len(SEEDS)} seeds, "
    "so the improvement is useful on average but not universal.\n\n"
    "The paper and project plan use a single-layer LSTM/BiLSTM as the exact baseline. "
    "The two-layer LSTM candidate is therefore reported as an ablation; it was not selected. "
    "LSTM+TCN is an additional hybrid architecture experiment, not a pretrained-model fine-tuning result.\n\n"
    "All candidate selection used the seed 5768 validation split only. "
    "The three held-out test sets were evaluated after freezing the selected configuration."
))


## 可复现检查清单

- 配置选择只使用 5768 validation，最终测试使用 5768、78516、944601。
- baseline、LSTM/BiLSTM 和数据处理实现都嵌入本 Notebook 的自包含代码单元。
- LSTM+TCN 和二阶段调参实现也嵌入本 Notebook，不依赖外部项目脚本。
- 每个最终 seed 都保存了配置、指标、classification report、原始和归一化混淆矩阵、预测、错误分析、学习曲线和 checkpoint。
- v2 结果目录是 results/lstm_three_paths_v2；之前的历史实验目录会在比较单元中保留并显示。

## 模型结构、规模与调参历史说明

### 1. 为什么先建立 baseline

论文和项目要求中的 LSTM/BiLSTM 是单层 recurrent model。我们先固定相同的数据读取、tokenisation、padding masking、cross-entropy、dropout、AdamW、scheduler、gradient clipping 和 early stopping，建立可比较的控制组。配置选择只使用 seed 5768 的 validation weighted F1；配置冻结后，才在 5768、78516、944601 三个 test seed 上评估。这样 test 结果不会反过来影响模型选择。

### 2. 三个模型的结构和大小

| 模型 | 实际结构 | 当前主要配置 | 参数量约数 | 规模判断 |
|---|---|---|---:|---|
| LSTM | 100 维 embedding → 单层 LSTM → final hidden representation → dropout → 3 类 linear classifier | hidden size 128，dropout 0.4 | 530k | 保持论文的单层结构，规模较小，适合作为严格 baseline |
| BiLSTM | 100 维 embedding → 单层双向 LSTM → max pooling → dropout → 3 类 linear classifier | 每个方向 hidden size 128，dropout 0.3 | 648k | 比 LSTM 多一些参数是双向输出的自然结果，但没有使用多层或超大 embedding |
| LSTM+TCN | 100 维 embedding → 单层 LSTM → 两个残差 TCN block → mean+max pooling → dropout → 3 类 linear classifier | channels 64，kernel 3，dilation 1/2，channel normalization | 601k | 只比 LSTM 稍大，仍是受控的额外 hybrid ablation，不是无限扩大模型 |

参数量会随每个 seed 的 vocabulary size 略有变化，因此表中的数字使用近似值。总体上，三种模型都没有采用两层 LSTM、巨大 embedding 或不受控的深层结构；模型规模和训练成本对当前句子分类任务是合理的。LSTM、BiLSTM 和 LSTM+TCN 的平均训练时间约为 19 秒、35 秒和 126 秒，TCN 的成本更高，但仍可在固定的 unsw 环境中重复运行。

### 3. 我们是怎样一步一步调参的

- **Baseline**：先复现单层 LSTM/BiLSTM，得到 LSTM `0.5005 ± 0.0169`、BiLSTM `0.4921 ± 0.0105` 的三 seed weighted F1。BiLSTM 的参数更多但没有自动更好，所以后面不能只凭模型复杂度判断结果。
- **第一轮 LSTM/BiLSTM tuning**：测试 dropout、hidden size、embedding size、batch size、learning rate、weight decay 和轻微 class weight。LSTM 选择 dropout50，但 test 均值降到 0.4873；BiLSTM 选择 embedding200，提升到 0.5066，但参数量增加到约 1.16M。这个阶段说明 validation 提升可能不能稳定转移到 test，也说明更大的 embedding 有成本。
- **LSTM v2**：围绕 dropout、pooling 和基础训练参数继续测试。dropout40 的 validation weighted F1 为 0.5797，但 test 均值为 0.4955，仍低于 baseline。
- **LSTM v3**：进一步检查 scheduler、weight decay、Adam、延长训练和 attention pooling。最好的候选仍然是原 control；因此我们没有继续增加 LSTM 层数。两层 LSTM 候选表现较差，也支持保留单层结构。
- **LSTM+TCN v1**：测试 channels、kernel、TCN 层数、pooling 和训练时间。单层 LSTM 加两层 TCN 的 longer-training 配置达到 `0.5097 ± 0.0301`，平均超过 LSTM baseline，但 seed 波动较大。
- **第二阶段 v2**：三条路径各测试 18 个受控候选。LSTM 选择 dropout40，BiLSTM 选择 max pooling，LSTM+TCN 选择 mean+max pooling 加 channel normalization。最终 BiLSTM v2 达到 `0.5281 ± 0.0166`，3/3 个 seed 超过对应 baseline；LSTM+TCN v2 达到 `0.5105 ± 0.0391`，平均有提升但稳定性仍弱；LSTM v2 没有超过 LSTM baseline。

### 4. 当前判断

当前结果不是所有改动都成功，而是 BiLSTM v2 找到了最清楚的改进。max pooling 可能帮助 BiLSTM 保留与政策立场相关的强词语或局部信号，但这只是基于结果的合理解释，不是额外的标签信息。LSTM+TCN 说明局部卷积特征有价值，不过其 weighted F1 标准差较大，所以 Notebook 中将它标为 additional hybrid architecture ablation，而不是替换论文 baseline。GloVe 因本地没有 100d 文件而跳过，没有下载外部数据。后续所有比较都应同时报告均值、标准差和 per-class F1，不能只挑最高的一次运行。

## 第二阶段三条路径实验（v2）

本节是当前只围绕 LSTM、BiLSTM 和 LSTM+TCN 的受控实验。候选配置只使用 seed 5768 的 validation split 选择；配置冻结后，再在 5768、78516、944601 三个 seed 上测试。主比较指标是 weighted F1 的均值和标准差，同时保留 macro F1、accuracy、三个类别的 F1、参数量和训练成本。

In [ ]:
# add
V2_ROOT = REPO_ROOT / "results" / "lstm_three_paths_v2"
V2_MODELS = {"lstm": "LSTM v2", "bilstm": "BiLSTM v2", "lstm_tcn": "LSTM+TCN v2"}
v2_comparison = pd.read_csv(V2_ROOT / "three_path_comparison.csv")
v2_columns = ["model", "baseline_reference", "weighted_f1_mean", "weighted_f1_std", "macro_f1_mean", "macro_f1_std", "accuracy_mean", "dovish_f1_mean", "hawkish_f1_mean", "neutral_f1_mean", "parameter_count_mean", "mean_training_seconds", "weighted_f1_delta_vs_baseline", "wins_vs_baseline", "status"]
display(Markdown("### 冻结配置的 v2 汇总"))
display(v2_comparison[v2_columns].round(4))
selected = json.loads((V2_ROOT / "validation_search" / "selected_configs.json").read_text())
selected_rows = [{"model": model, **config} for model, config in selected["configs"].items()]
display(Markdown("### 只在 validation seed 5768 上选择的配置"))
display(pd.DataFrame(selected_rows).T)
search = pd.read_csv(V2_ROOT / "validation_search" / "three_paths_validation_search.csv")
search_columns = ["model", "candidate", "validation_weighted_f1", "validation_macro_f1", "parameter_count", "attempt_status"]
display(Markdown("### 完整候选历史，包括失败和未选中的候选"))
display(search[search_columns].sort_values(["model", "validation_weighted_f1"], ascending=[True, False]).round(4))
glove_status = json.loads((V2_ROOT / "glove_status.json").read_text())
display(Markdown("GloVe 状态：" + glove_status["status"] + "。" + glove_status["reason"]))
#done


In [ ]:
# add
v2_final = pd.read_csv(V2_ROOT / "final" / "lstm_bilstm_summary.csv")
display(Markdown("### 冻结模型的逐 seed 证据"))
display(v2_final.round(4))
# add
for figure_name in ["three_paths_weighted_f1.png", "three_paths_macro_f1.png", "three_paths_per_class_f1.png", "three_paths_cost.png"]:
    show_image(V2_ROOT / figure_name)
#done
for model, title in V2_MODELS.items():
    display(Markdown(f"#### {title}"))
    for seed in SEEDS:
        run_dir = V2_ROOT / "final" / model / f"seed_{seed}"
        metrics = json.loads((run_dir / "metrics.json").read_text())
        test_metrics = metrics.get("test", metrics)
        display(Markdown(f"Seed {seed}：weighted F1 = {test_metrics['weighted_f1']:.4f}，macro F1 = {test_metrics['macro_f1']:.4f}，accuracy = {test_metrics['accuracy']:.4f}"))
        display(pd.read_csv(run_dir / "classification_report.csv").round(4))
        display(pd.read_csv(run_dir / "confusion_matrix_raw.csv"))
        show_image(run_dir / "confusion_matrix_normalized.png")
        show_image(run_dir / "learning_curve.png")
        error_path = run_dir / "selected_error_cases.csv"
        if error_path.exists():
            display(pd.read_csv(error_path).head(20))
#done
